# PREPROCESSING SCRIPT AFRICA - STAC version

Same workflow as `Combined_Preprocessing_AFRICA_VH.ipynb` (same 8 sections, same formulas, same outputs), but **the input data is read from the public STAC catalog** (`gca-stac-7`) instead of from local files. Everything that is created is saved in

`P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\model`

using the same folder layout as the original (`Data\1_QGIS_Processed\...` and `Modeling\AFRICA\IMOD\Inputs\...`).

Sections: 0) setup, 1) HK soil (SoilGrids + Cosby), 2) GST raster, 3) data harmonization, 4) HK consolidated (GLiM), 5) HK unconsolidated (GUM), 6) porosity, 7) numerical layers, 8) RIV package.

Environment: `environment_exploration.yml` (`mamba env update -n gca-exploration -f environment_exploration.yml`).

### Inputs and where they come from

| Input | Used in | Source |
|---|---|---|
| SoilGrids clay and sand, 6 depths | 1, 6 | STAC `soil_clay_cog`, `soil_sand_cog` |
| Global sediment thickness (GST) | 2, 3, 7 | STAC `gst` (global 0.125° COG, replaces the XYZ points) |
| Soil thickness (depth to bedrock) | 3 | STAC `soil_depth` (global, clipped to Africa here) |
| GEBCO 2025 | 3, 8 | STAC `gebco` (4 tiles, mosaicked and clipped to Africa here) |
| Groundwater recharge (RF_RK) | 3 | STAC `groundwater_recharge` |
| HydroLAKES | 3 | STAC `hydrolakes` |
| GLHYMPS 2.0 | 3, 6 | STAC `glhymps` |
| GLiM | 4, 5, 6 | STAC `glim` |
| GUM | 5 | STAC `gum` |
| Seabed lithology | 7 | STAC `seabed_lithology` |
| HydroRIVERS | 8 | STAC `hydrorivers` |
| **Model domain polygon** | 3, 7, 8 | **NOT in the STAC** |
| **Inland polygon** | 3, 5, 7, 8 | **NOT in the STAC** |
| **Unconsolidated sediment thickness** | 3, 7 | **NOT in the STAC** |
| **Aridity index (ai_v31_yr.tif)** | 3, 8 | **NOT in the STAC** |

### Missing inputs

The four inputs in bold are not in the STAC. Put them in the folder `Data\missing_inputs` (next to the other results, see the setup cell for the exact path and the expected file names). The notebook runs without them: every output that needs one of them is **skipped with a "NOT CREATED" message**, and the last cell lists what was created and what is still missing. Once the files are in place, run the notebook again.

### 0) Setup: folders, STAC access and helper functions

In [ ]:
import os
import time
import shutil
import warnings

import fsspec
import geopandas as gpd
import numpy as np
import pystac_client
import rasterio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.warp import reproject, transform_bounds
from rasterio.windows import Window
from shapely.geometry import box

warnings.filterwarnings("ignore", module="pystac_client")

# ==========================================================================
# FOLDERS (same layout as the original: ...\THESIS\Data and ...\THESIS\Modeling)
# ==========================================================================
MODEL_DIR = r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\model"

PROCESSED_DIR = os.path.join(MODEL_DIR, "Data", "1_QGIS_Processed")        # was ...\Data\1_QGIS_Processed
MISSING_DIR = os.path.join(MODEL_DIR, "Data", "missing_inputs")            # inputs that are not in the STAC
INPUTS_5KM = os.path.join(MODEL_DIR, "Modeling", "AFRICA", "IMOD", "Inputs", "5_x_5_km")
INPUTS_10KM = os.path.join(MODEL_DIR, "Modeling", "AFRICA", "IMOD", "Inputs", "10_x_10_km")
ML6_DIR = os.path.join(INPUTS_10KM, "6_ml")
RIVER_PLOT_DIR = os.path.join(MODEL_DIR, "Modeling", "AFRICA", "Outputs", "10_x_10_km", "river_package")

for folder in [MISSING_DIR, INPUTS_5KM, INPUTS_10KM, ML6_DIR, RIVER_PLOT_DIR] + [
    os.path.join(PROCESSED_DIR, sub) for sub in ["unharmonized", "SOILGRIDS", "GST", "GLIM", "GUM", "GLHYMPS"]
]:
    os.makedirs(folder, exist_ok=True)

# True: download/convert the STAC data again even if the local copy already exists
REFRESH_LOCAL_COPIES = False

# ==========================================================================
# MODEL GRID AND READING AREA
# ==========================================================================
TARGET_CRS = "EPSG:3857"
MODEL_EXTENT = (-2145000, -4440000, 5965000, 4645000)   # xmin, ymin, xmax, ymax (EPSG:3857)

west, south, east, north = transform_bounds(TARGET_CRS, "EPSG:4326", *MODEL_EXTENT)
# model extent in lon/lat plus a margin, on whole degrees: only this area is read from the STAC
AFRICA_BBOX = (float(np.floor(west) - 1), float(np.floor(south) - 1), float(np.ceil(east) + 1), float(np.ceil(north) + 1))
print("Reading area (west, south, east, north):", AFRICA_BBOX)

# ==========================================================================
# INPUTS THAT ARE NOT IN THE STAC (put them in MISSING_DIR)
# ==========================================================================
MISSING_FILES = {
    "domain": ("Africa_domain_epsg3857.shp", "Model domain polygon"),
    "inland": ("Africa_inland.shp", "Inland (land) polygon, land/ocean mask"),
    "unconsolidated": ("_WORLD_SU_thk_est_mean_bestfit_thk_mWORLD_BESTFIT.tif", "Unconsolidated sediment thickness (m)"),
    "aridity": ("ai_v31_yr.tif", "Global Aridity Index v3.1 (values x 10000)"),
}
MISSING_PATHS = {key: os.path.join(MISSING_DIR, name) for key, (name, _) in MISSING_FILES.items()}


def have(*names):
    """True if all the given inputs that are not in the STAC are present in MISSING_DIR."""
    return all(os.path.exists(MISSING_PATHS[n]) for n in names)


def require(output, *names):
    """Check the inputs needed for an output; print why it is skipped if one is missing."""
    missing = [MISSING_FILES[n][0] for n in names if not have(n)]
    if missing:
        print(f"NOT CREATED: {output}  <- missing input: {', '.join(missing)}")
        return False
    return True


print("\nInputs that are not in the STAC (folder:", MISSING_DIR + ")")
for key, (name, description) in MISSING_FILES.items():
    print(f"  {'found  ' if have(key) else 'MISSING'}  {name}  ({description})")

# ==========================================================================
# STAC ACCESS
# ==========================================================================
GCS = "https://storage.googleapis.com"
CATALOG_URL = f"{GCS}/gca-data-public/gca/gca-stac-7/catalog.json"
# the query string forces a fresh copy (public objects are cached for 1 h)
catalog = pystac_client.Client.open(f"{CATALOG_URL}?nocache={time.time()}")


def get_items(collection_id):
    """Return {item id: pystac.Item} of a collection of the catalog."""
    collection = next((c for c in catalog.get_children() if c.id == collection_id), None)
    if collection is None:
        raise LookupError(f"Collection {collection_id} not found in the catalog")
    return {i.id: i for i in collection.get_items()}


def stac_href(collection_id, item_id=None):
    """URL of the `data` asset of an item (the first item if item_id is not given)."""
    items = get_items(collection_id)
    item = items[item_id] if item_id else next(iter(items.values()))
    return item.assets["data"].href


def read_stac_vector(collection_id, bbox=AFRICA_BBOX):
    """Read a GeoParquet collection of the STAC; only the polygons/lines inside bbox are downloaded."""
    href = stac_href(collection_id)
    gdf = gpd.read_parquet(href, bbox=bbox, filesystem=fsspec.filesystem("https"))
    gdf = gdf.drop(columns="bbox", errors="ignore")
    print(f"STAC {collection_id}: {len(gdf):,} features | CRS {gdf.crs}")
    return gdf


def read_cog_window(href, bbox, decimate=1):
    """
    Read the part of an EPSG:4326 COG that lies inside bbox (west, south, east, north).
    decimate > 1 reads a coarser version (the COG overviews are used, so little data is downloaded).
    Returns the array, its transform and the nodata value.
    """
    eps = 1e-6
    with rasterio.open(href) as src:
        t = src.transform
        col0 = max(int(np.floor((bbox[0] - t.c) / t.a + eps)), 0)
        col1 = min(int(np.ceil((bbox[2] - t.c) / t.a - eps)), src.width)
        row0 = max(int(np.floor((bbox[3] - t.f) / t.e + eps)), 0)
        row1 = min(int(np.ceil((bbox[1] - t.f) / t.e - eps)), src.height)
        window = Window(col0, row0, col1 - col0, row1 - row0)
        out_shape = (int(window.height // decimate), int(window.width // decimate))
        data = src.read(1, window=window, out_shape=out_shape, resampling=Resampling.average)
        transform = src.window_transform(window) * Affine.scale(
            window.width / out_shape[1], window.height / out_shape[0]
        )
        return data, transform, src.nodata


def write_tif(path, array, transform, crs, nodata):
    with rasterio.open(
        path, "w", driver="GTiff", height=array.shape[0], width=array.shape[1], count=1,
        dtype=array.dtype, crs=crs, transform=transform, nodata=nodata, compress="lzw",
    ) as dst:
        dst.write(array, 1)
    print("Saved:", path)

### 1) Assign HK to GEO UNIT "Soil", based on SOILGRIDS Soil Texture + PTF

Here we use the Soilgrids soil texture data to translate it via a pedotransfer function (PTF) into hydraulic conductivity values [m/day]. In this case the Cosby function is used because it uses the sand fraction, but any better fitting function can be integrated and tested based on the fractions of sand, silt and clay that are calculated in this script.

**Source:** STAC collections `soil_clay_cog` and `soil_sand_cog` (6 items each, same names as the original files: `clay_5km_0-5cm` ... `sand_5km_100-200cm`). The rasters are read straight from the catalog. The result is saved in `Data\1_QGIS_Processed\SOILGRIDS`.

In [ ]:
# This script processes the clay and sand content rasters into a weighted average per cell
SOILGRIDS_DIR = os.path.join(PROCESSED_DIR, "SOILGRIDS")

# model extent (EPSG:3857)
xmin, ymin, xmax, ymax = -2145000, -4440000, 5965000, 4645000

cell_size = 5000  # 5 km

# depth weights (meters)
depths = [
    ("0-5cm", 0.05),
    ("5-15cm", 0.10),
    ("15-30cm", 0.15),
    ("30-60cm", 0.30),
    ("60-100cm", 0.40),
    ("100-200cm", 1.00),
]

nodata_value = -9999

# CREATE TARGET GRID
width = int((xmax - xmin) / cell_size)
height = int((ymax - ymin) / cell_size)

transform = rasterio.transform.from_origin(xmin, ymax, cell_size, cell_size)


def weighted_depth_average(variable, output_file):
    """Depth-weighted average of the 6 SoilGrids layers of `variable` (clay or sand) on the 5 km model grid."""
    items = get_items(f"soil_{variable}_cog")

    # containers
    weighted_sum = np.zeros((height, width), dtype=np.float32)
    weight_total = np.zeros((height, width), dtype=np.float32)

    # PROCESS EACH DEPTH
    for depth_name, thickness in depths:
        print(f"Processing {variable}, depth: {depth_name}")

        href = items[f"{variable}_5km_{depth_name}"].assets["data"].href

        with rasterio.open(href) as src:

            # prepare destination array
            dest = np.zeros((height, width), dtype=np.float32)

            # reproject + resample to model grid
            reproject(
                source=rasterio.band(src, 1),
                destination=dest,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs="EPSG:3857",
                resampling=Resampling.average
            )

            # mask nodata
            if src.nodata is not None:
                dest = np.where(dest == src.nodata, np.nan, dest)

            # accumulate weighted sum
            weighted_sum += dest * thickness
            weight_total += thickness

    # FINAL WEIGHTED AVERAGE
    final = weighted_sum / weight_total

    # set NaN to nodata
    final = np.where(np.isnan(final), nodata_value, final)

    # SAVE OUTPUT
    with rasterio.open(
        output_file,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="float32",
        crs="EPSG:3857",
        transform=transform,
        nodata=nodata_value,
    ) as dst:
        dst.write(final, 1)

    print("Done. Output saved to:", output_file)


# ========================================= CLAY ================================================
weighted_depth_average("clay", os.path.join(SOILGRIDS_DIR, "clay_weighted_5km_3857.tif"))

# ========================================= SAND ================================================
weighted_depth_average("sand", os.path.join(SOILGRIDS_DIR, "sand_weighted_5km_3857.tif"))

In [ ]:
# ================================= COMPUTE SILT CONTENT IN % ====================================
sand_file = os.path.join(SOILGRIDS_DIR, "sand_weighted_5km_3857.tif")
clay_file = os.path.join(SOILGRIDS_DIR, "clay_weighted_5km_3857.tif")

output_file = os.path.join(SOILGRIDS_DIR, "silt_weighted_5km_3857.tif")

nodata_value = -9999

# LOAD INPUT RASTERS
with rasterio.open(sand_file) as sand_src, rasterio.open(clay_file) as clay_src:

    # sanity check: grids must match
    assert sand_src.shape == clay_src.shape, "Raster shapes do not match"
    assert sand_src.transform == clay_src.transform, "Transforms do not match"
    assert sand_src.crs == clay_src.crs, "CRS do not match"

    sand = sand_src.read(1).astype(np.float32)
    clay = clay_src.read(1).astype(np.float32)

    profile = sand_src.profile

# HANDLE NODATA
sand = np.where(sand == sand_src.nodata, np.nan, sand)
clay = np.where(clay == clay_src.nodata, np.nan, clay)

# COMPUTE SILT
silt = 100.0 - sand - clay

# enforce bounds
silt = np.clip(silt, 0, 100)

# where inputs are nan -> output nan
mask = np.isnan(sand) | np.isnan(clay)
silt[mask] = np.nan

# WRITE OUTPUT
silt_out = np.where(np.isnan(silt), nodata_value, silt)

profile.update(dtype="float32", nodata=nodata_value)

with rasterio.open(output_file, "w", **profile) as dst:
    dst.write(silt_out.astype(np.float32), 1)

print("Done. Silt raster saved to:", output_file)

In [ ]:
#====================================================================================
#                            ASSIGN HK VIA COSBY FUNCTION
#====================================================================================
sand_file = os.path.join(SOILGRIDS_DIR, "sand_weighted_5km_3857.tif")
output_file = os.path.join(SOILGRIDS_DIR, "hk_soil_cosby_5km.tif")

nodata_value = -9999

# LOAD INPUT RASTER
with rasterio.open(sand_file) as src:
    sand = src.read(1).astype(np.float32)
    profile = src.profile

# HANDLE NODATA
sand = np.where(sand == src.nodata, np.nan, sand)

# COSBY EQUATION
# Ksat = 0.0070556 * 10^(-0.884 + 0.0153 * sand%)
# Units: cm/s
K_cm_per_s = 0.0070556 * (10 ** (-0.884 + 0.0153 * sand))

# CONVERT TO m/day
# cm/s -> m/s -> m/day
K_m_per_day = K_cm_per_s * 0.01 * 86400.0

# CLEANUP / STABILITY
# remove unrealistic extremes
K_m_per_day = np.clip(K_m_per_day, 1e-5, 100)

# apply nodata mask
K_m_per_day[np.isnan(sand)] = np.nan

# WRITE OUTPUT
K_out = np.where(np.isnan(K_m_per_day), nodata_value, K_m_per_day)

profile.update(dtype="float32", nodata=nodata_value)

with rasterio.open(output_file, "w", **profile) as dst:
    dst.write(K_out.astype(np.float32), 1)

print("Done. Cosby hydraulic conductivity saved to:", output_file)

### 2) GST (GLOBAL SEDIMENT THICKNESS) -> RASTER DATA

The original script converted the XYZ points (Bird & Mooney) into a raster. In the STAC the same data is already a raster: collection `gst`, a global 0.125° COG (`gst.tif`, nodata -9999). This section only **reads the part that covers Africa** and saves it as `GST_python_epsg4326.tif`, the file that the harmonization (section 3) expects.

Note: in the STAC raster every XYZ point is the **centre** of a 0.125° cell (the grid origin is -180.0625, 90.0625). The original script used the point as the corner of the cell, so the STAC version is shifted by half a cell (0.0625°, about 7 km) compared to the original GST raster.

In [ ]:
# STAC raster -> Africa extract (replaces the points -> raster conversion)
gst_href = stac_href("gst")
print("GST COG:", gst_href)

raster, gst_transform, gst_nodata = read_cog_window(gst_href, AFRICA_BBOX)
raster = raster.astype(np.float32)

# --- CHECKS ---
valid = raster != gst_nodata
print(f"Raster size: {raster.shape[1]} x {raster.shape[0]}")
print(f"Missing cells: {int((~valid).sum())}")
print(f"Min value: {raster[valid].min()}")
print(f"Max value: {raster[valid].max()}")

# --- WRITE GEOTIFF ---
output_file = os.path.join(PROCESSED_DIR, "GST", "GST_python_epsg4326.tif")
write_tif(output_file, raster, gst_transform, "EPSG:4326", -9999)

### 3) DATA HARMONIZATION

This is the longest section, combined of many individual preprocessing scripts, that all include either rasterization, resampling, reprojection, no data handling, unit conversion, clipping, etc.

Data that is harmonized here: GEBCO, soil thickness, recharge, GST, unconsolidated sediment thickness, soil hydraulic conductivity (HK), aridity index, domain and inland masks, lakes, GLHYMPS (porosity).

**Differences with the original**
- `Africa_GEBCO.tif` and `Africa_SOIL.tif` (clipped manually in QGIS in the original) are created here from the STAC GEBCO tiles and the global soil depth COG, at about 1 km (30 arc-seconds), and saved in `unharmonized`. `Africa_GEBCO_3857.tif` (1 km, EPSG:3857) is also created here: section 8 needs it.
- Recharge is read directly from the STAC (`RF_RK_recharge_01`), lakes and GLHYMPS from the STAC GeoParquet files.
- **Not in the STAC** (see the table at the top): domain polygon, inland polygon, unconsolidated thickness, aridity index. Every output that needs them is skipped (`NOT CREATED`). Without the inland polygon `soil_thickness_10km`, `recharge_10km` and `hk_soil_10km` cannot be created, without the domain polygon `gst_10km`, `lakes_10km`, `unconsolidated_10km` and `aridity_index_10km` cannot be created. Only `gebco_10km.tif` and the GLHYMPS GeoPackage need no missing input.

In [ ]:
# ============================================================
# Harmonizes raster and vector inputs to a shared 10 km grid.
# ============================================================
# SHARED MODEL GRID
TARGET_CRS = "EPSG:3857"
RES = 10_000  # 10 km

XMIN, YMIN, XMAX, YMAX = MODEL_EXTENT

WIDTH = int((XMAX - XMIN) / RES)
HEIGHT = int((YMAX - YMIN) / RES)

TRANSFORM = from_origin(
    XMIN,
    YMAX,
    RES,
    RES
)

# Common intermediate raster nodata value
RAW_NODATA = -9999.0

# Final continuous raster nodata value
FINAL_NODATA = -999.0

# OUTPUT
OUT_DIR = INPUTS_10KM

print("Model grid")
print("----------------------------")
print(f"CRS:       {TARGET_CRS}")
print(f"Resolution: {RES} m")
print(f"Width:      {WIDTH} columns")
print(f"Height:     {HEIGHT} rows")
print(f"Pixel size: {RES} x {RES} m")
print(f"Actual X bounds: {XMIN} to {XMIN + WIDTH * RES}")
print(f"Actual Y bounds: {YMAX - HEIGHT * RES} to {YMAX}")


# ============================================================
# COMMON RASTER HARMONIZATION FUNCTION
# ============================================================

def reproject_to_grid(
    input_path,
    output_path,
    name,
    resampling=Resampling.bilinear,
    dst_nodata=RAW_NODATA
):
    """
    Reproject a single-band raster (local file or URL) onto the fixed model grid.

    Dataset-specific resampling is selected when calling
    the function. Source nodata values and NaNs are handled
    before reprojection.
    """

    print(f"\nReprojecting: {name}")

    with rasterio.open(input_path) as src:

        if src.crs is None:
            raise ValueError(
                f"{name} has no CRS: {input_path}"
            )

        if src.count != 1:
            raise ValueError(
                f"{name} has {src.count} bands. "
                "This function expects a single-band raster."
            )

        src_data = src.read(1).astype(np.float32)

        # Preserve the source nodata definition.
        src_nodata = src.nodata

        # Ensure NaNs are represented by a nodata value.
        if src_nodata is None:
            src_nodata = RAW_NODATA

        src_data[~np.isfinite(src_data)] = src_nodata

        # Destination grid
        dst_data = np.full(
            (HEIGHT, WIDTH),
            dst_nodata,
            dtype=np.float32
        )

        reproject(
            source=src_data,
            destination=dst_data,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,
            dst_transform=TRANSFORM,
            dst_crs=TARGET_CRS,
            dst_nodata=dst_nodata,
            resampling=resampling
        )

    with rasterio.open(
        output_path,
        "w",
        driver="GTiff",
        height=HEIGHT,
        width=WIDTH,
        count=1,
        dtype="float32",
        crs=TARGET_CRS,
        transform=TRANSFORM,
        nodata=dst_nodata,
        compress="lzw"
    ) as dst:
        dst.write(dst_data, 1)

    valid = dst_data[
        np.isfinite(dst_data) & (dst_data != dst_nodata)
    ]

    print(f"Saved: {output_path}")
    print(f"Valid cells: {valid.size:,}")

    if valid.size:
        print(f"Min: {valid.min():.4f}")
        print(f"Max: {valid.max():.4f}")

    return output_path


# ============================================================
# COMMON VECTOR FUNCTIONS
# ============================================================

def load_vector(source, name):
    """Load a vector dataset (file path or GeoDataFrame) and reproject it to the model CRS."""

    print(f"\nLoading {name}...")

    gdf = gpd.read_file(source) if isinstance(source, str) else source

    if gdf.crs is None:
        raise ValueError(
            f"{name} has no CRS defined: {source}"
        )

    gdf = gdf.to_crs(TARGET_CRS)

    # Remove missing or empty geometries.
    gdf = gdf.loc[
        gdf.geometry.notna() & ~gdf.geometry.is_empty
    ].copy()

    print(f"Features: {len(gdf):,}")

    return gdf


def rasterize_vector(
    gdf,
    name,
    output_path,
    fill=0,
    burn_value=1,
    dtype="uint8",
    nodata=255
):
    """
    Rasterize a vector dataset on the shared model grid.

        1 = inside the geometry
        0 = outside the geometry

    Nodata is kept separate from the valid binary values.
    """

    print(f"\nRasterizing {name}...")

    shapes = [
        (geom, burn_value)
        for geom in gdf.geometry
        if geom is not None and not geom.is_empty
    ]

    arr = rasterize(
        shapes,
        out_shape=(HEIGHT, WIDTH),
        transform=TRANSFORM,
        fill=fill,
        dtype=dtype
    )

    with rasterio.open(
        output_path,
        "w",
        driver="GTiff",
        height=HEIGHT,
        width=WIDTH,
        count=1,
        dtype=dtype,
        crs=TARGET_CRS,
        transform=TRANSFORM,
        nodata=nodata,
        compress="lzw"
    ) as dst:
        dst.write(arr, 1)

    print(f"Saved: {output_path}")
    print(f"Unique values: {np.unique(arr)}")

    return arr

#### 3a) Africa extracts of GEBCO and soil thickness (STAC)

GEBCO is split in 4 tiles (21600 x 21600 pixels each): only the Africa window is read from each tile at half resolution (30 arc-seconds, the COG overviews are used) and mosaicked. The soil depth COG is global at 7.5 arc-seconds: it is read at 30 arc-seconds. Both are saved as GeoTIFF in `unharmonized`, like the files that were clipped manually in the original workflow.

In [ ]:
UNHARM_DIR = os.path.join(PROCESSED_DIR, "unharmonized")
GEBCO_PATH = os.path.join(UNHARM_DIR, "Africa_GEBCO.tif")             # EPSG:4326, ~1 km
GEBCO_3857_PATH = os.path.join(UNHARM_DIR, "Africa_GEBCO_3857.tif")   # EPSG:3857, 1 km (used in section 8)
SOIL_PATH = os.path.join(UNHARM_DIR, "Africa_SOIL.tif")               # EPSG:4326, ~1 km, cm

GEBCO_DECIMATE = 2   # 15 arc-sec x 2 = 30 arc-sec
SOIL_DECIMATE = 4    # 7.5 arc-sec x 4 = 30 arc-sec
res_out = 1 / 120    # 30 arc-sec in degrees
W, S, E, N = AFRICA_BBOX

# ------------------------------- GEBCO (4 tiles -> one Africa raster) -------------------------------
if REFRESH_LOCAL_COPIES or not os.path.exists(GEBCO_PATH):
    ncols = int(round((E - W) / res_out))
    nrows = int(round((N - S) / res_out))
    mosaic = np.full((nrows, ncols), -32767, dtype="int16")

    for item_id, item in get_items("gebco").items():
        b = item.bbox
        if b[2] <= W or b[0] >= E or b[3] <= S or b[1] >= N:
            continue  # tile outside the reading area
        data, tr, nodata = read_cog_window(item.assets["data"].href, AFRICA_BBOX, GEBCO_DECIMATE)
        c0 = int(round((tr.c - W) / res_out))
        r0 = int(round((N - tr.f) / res_out))
        mosaic[r0:r0 + data.shape[0], c0:c0 + data.shape[1]] = data
        print(f"{item_id}: {data.shape}")

    write_tif(GEBCO_PATH, mosaic, from_origin(W, N, res_out, res_out), "EPSG:4326", -32767)
    del mosaic
else:
    print("Exists:", GEBCO_PATH)

# ------------------------------- GEBCO 1 km in EPSG:3857 -------------------------------
if REFRESH_LOCAL_COPIES or not os.path.exists(GEBCO_3857_PATH):
    res_1km = 1000
    width_1km = int((XMAX - XMIN) / res_1km)
    height_1km = int((YMAX - YMIN) / res_1km)
    transform_1km = from_origin(XMIN, YMAX, res_1km, res_1km)

    dst_1km = np.full((height_1km, width_1km), -9999, dtype="float32")
    with rasterio.open(GEBCO_PATH) as src:
        reproject(
            source=src.read(1).astype("float32"),
            destination=dst_1km,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=transform_1km,
            dst_crs=TARGET_CRS,
            dst_nodata=-9999,
            resampling=Resampling.bilinear,
        )
    write_tif(GEBCO_3857_PATH, dst_1km, transform_1km, TARGET_CRS, -9999)
    del dst_1km
else:
    print("Exists:", GEBCO_3857_PATH)

# ------------------------------- Soil thickness (cm) -------------------------------
if REFRESH_LOCAL_COPIES or not os.path.exists(SOIL_PATH):
    soil_href = stac_href("soil_depth")
    print("Soil depth COG:", soil_href)
    data, tr, nodata = read_cog_window(soil_href, AFRICA_BBOX, SOIL_DECIMATE)
    write_tif(SOIL_PATH, data, tr, "EPSG:4326", nodata)
    del data
else:
    print("Exists:", SOIL_PATH)

#### 3b) Domain and inland masks

Both polygons are **not in the STAC**: they are read from `Data\missing_inputs`.

In [ ]:
# ============================================================
# LOAD DOMAIN AND INLAND POLYGONS AND CREATE THE MASKS
# ============================================================
DOMAIN_SHP = MISSING_PATHS["domain"]
INLAND_SHP = MISSING_PATHS["inland"]

domain_mask_path = os.path.join(OUT_DIR, "domain_10km.tif")
inland_mask_path = os.path.join(OUT_DIR, "inland_10km.tif")

domain = None
domain_mask_array = None
inland = None
inland_mask_array = None

if require("domain_10km.tif", "domain"):
    domain = load_vector(DOMAIN_SHP, "Model domain")
    domain_mask_array = rasterize_vector(
        domain,
        "Domain mask",
        domain_mask_path
    )

if require("inland_10km.tif", "inland"):
    inland = load_vector(INLAND_SHP, "Inland polygons")
    inland_mask_array = rasterize_vector(
        inland,
        "Inland mask",
        inland_mask_path
    )

#### 3c) Lakes (HydroLAKES, STAC) and GLHYMPS (STAC)

In [ ]:
# ============================================================
# PROCESS LAKES
# Reproject -> intersect with domain -> rasterize
# ============================================================
lakes_path = os.path.join(OUT_DIR, "lakes_10km.tif")

if require("lakes_10km.tif", "domain"):
    lakes = load_vector(read_stac_vector("hydrolakes"), "Lakes (STAC hydrolakes)")

    print("\nClipping lakes to model domain...")

    lakes = gpd.overlay(
        lakes,
        domain[["geometry"]],
        how="intersection"
    )

    # Remove empty geometries resulting from the intersection.
    lakes = lakes.loc[
        lakes.geometry.notna() & ~lakes.geometry.is_empty
    ].copy()

    print(f"Remaining lake polygons: {len(lakes):,}")

    lake_array = rasterize_vector(
        lakes,
        "Lakes",
        lakes_path
    )

    print("Lake cells:", int(np.sum(lake_array == 1)))
    print("Non-lake cells:", int(np.sum(lake_array == 0)))

In [ ]:
# ============================================================
# PROCESS GLHYMPS
# Reproject -> bounding-box crop -> save vector
# ============================================================
glhymps = load_vector(read_stac_vector("glhymps"), "GLHYMPS (STAC glhymps)")

print("\nCropping GLHYMPS to model grid extent...")

# Preserve the original GLHYMPS workflow: bounding-box crop.
# This is NOT a polygon intersection with the domain boundary.
glhymps_clip = glhymps.cx[
    XMIN:XMIN + WIDTH * RES,
    YMAX - HEIGHT * RES:YMAX
].copy()

print(f"Remaining GLHYMPS features: {len(glhymps_clip):,}")

# GeoPackage avoids the field-name and file-management
# limitations of the Shapefile format.
glhymps_output = os.path.join(
    OUT_DIR,
    "GLHYMPS_Africa_3857.gpkg"
)

if os.path.exists(glhymps_output):
    os.remove(glhymps_output)

glhymps_clip.to_file(
    glhymps_output,
    driver="GPKG"
)

print("Saved:", glhymps_output)
print("CRS:", glhymps_clip.crs)
print("Extent:", glhymps_clip.total_bounds)

#### 3d) Reproject the continuous rasters to the 10 km grid

Each dataset keeps its own resampling method (GEBCO, soil thickness, recharge, unconsolidated thickness, aridity: bilinear; GST: nearest; HK soil: average). The unconsolidated thickness and the aridity index are **not in the STAC**.

In [ ]:
# Each dataset keeps its own resampling method.
# All outputs use the same dimensions, CRS, and transform.

# RASTERS
RECHARGE_PATH = stac_href("groundwater_recharge", "RF_RK_recharge_01")             # STAC (read from the URL)
ARIDITY_PATH = MISSING_PATHS["aridity"]                                            # NOT in the STAC
UNCONSOLIDATED_PATH = MISSING_PATHS["unconsolidated"]                              # NOT in the STAC
GST_PATH = os.path.join(PROCESSED_DIR, "GST", "GST_python_epsg4326.tif")           # section 2
HK_SOIL_PATH = os.path.join(PROCESSED_DIR, "SOILGRIDS", "hk_soil_cosby_5km.tif")   # section 1
# GEBCO_PATH and SOIL_PATH: Africa extracts created in 3a

raw_paths = {}

# GEBCO: bilinear ----------------------------
raw_paths["gebco"] = os.path.join(OUT_DIR, "gebco_10km_raw.tif")
reproject_to_grid(GEBCO_PATH, raw_paths["gebco"], "GEBCO", resampling=Resampling.bilinear)

# Soil thickness: bilinear ---------------------
raw_paths["soil"] = os.path.join(OUT_DIR, "soil_10km_raw.tif")
reproject_to_grid(SOIL_PATH, raw_paths["soil"], "Soil thickness", resampling=Resampling.bilinear)

# Recharge: bilinear ----------------------------
raw_paths["recharge"] = os.path.join(OUT_DIR, "recharge_10km_raw.tif")
reproject_to_grid(RECHARGE_PATH, raw_paths["recharge"], "Recharge", resampling=Resampling.bilinear)

# GST: nearest-neighbour -------------------------
# Replaces the separate GST reprojection script.
raw_paths["gst"] = os.path.join(OUT_DIR, "gst_10km_raw.tif")
reproject_to_grid(GST_PATH, raw_paths["gst"], "GST", resampling=Resampling.nearest)

# Soil hydraulic conductivity: --------------------
# average resampling from the 5 km source to the 10 km grid.
# Replaces the separate HK resampling script.
raw_paths["hk_soil"] = os.path.join(OUT_DIR, "hk_soil_10km_raw.tif")
reproject_to_grid(HK_SOIL_PATH, raw_paths["hk_soil"], "Soil hydraulic conductivity", resampling=Resampling.average)

# Unconsolidated sediment thickness: bilinear (NOT in the STAC) -----
raw_paths["unconsolidated"] = os.path.join(OUT_DIR, "unconsolidated_10km_raw.tif")
if require("unconsolidated_10km_raw.tif", "unconsolidated"):
    reproject_to_grid(
        UNCONSOLIDATED_PATH, raw_paths["unconsolidated"], "Unconsolidated sediment thickness",
        resampling=Resampling.bilinear
    )

# Aridity index: bilinear (NOT in the STAC) --------------------------
raw_paths["aridity"] = os.path.join(OUT_DIR, "aridity_index_10km_raw.tif")
if require("aridity_index_10km_raw.tif", "aridity"):
    reproject_to_grid(ARIDITY_PATH, raw_paths["aridity"], "Aridity index", resampling=Resampling.bilinear)

print("\nRaster reprojections completed.")

#### 3e) Clean, convert units, apply masks and save the final rasters

Same processing as the original (cm -> m for soil thickness, NaN -> 0, negative thickness -> 0, inland/domain masks, HK only where soil > 0, recharge mm/year -> m/day, aridity / 10000). Outputs whose mask or input is missing are skipped.

In [ ]:
# ============================================================
# LOAD HARMONIZED RASTERS
# ============================================================

def load_raw_array(path):
    """
    Load the single-band harmonized raster.
    Convert declared nodata values to NaN for processing.
    """

    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata

    if nodata is not None:
        if np.isnan(nodata):
            arr[~np.isfinite(arr)] = np.nan
        else:
            arr[arr == nodata] = np.nan

    arr[~np.isfinite(arr)] = np.nan

    return arr


gebco = load_raw_array(raw_paths["gebco"])
soil = load_raw_array(raw_paths["soil"])            # soil depth
recharge = load_raw_array(raw_paths["recharge"])
gst = load_raw_array(raw_paths["gst"])
hk_soil = load_raw_array(raw_paths["hk_soil"])
unconsolidated = load_raw_array(raw_paths["unconsolidated"]) if have("unconsolidated") else None
aridity = load_raw_array(raw_paths["aridity"]) if have("aridity") else None

# ============================================================
# SOIL UNIT CONVERSION: cm -> m
# ============================================================
soil = soil / 100.0

# ============================================================
# CLEAN MISSING VALUES
# ============================================================
# GEBCO, soil depth, recharge, GST, and unconsolidated sediment
# missing values are replaced with zero.
# HK retains NaN to distinguish missing data from zero HK.

gebco = np.nan_to_num(gebco, nan=0.0, posinf=0.0, neginf=0.0)
soil = np.nan_to_num(soil, nan=0.0, posinf=0.0, neginf=0.0)
recharge = np.nan_to_num(recharge, nan=0.0, posinf=0.0, neginf=0.0)
gst = np.nan_to_num(gst, nan=0.0, posinf=0.0, neginf=0.0)
if unconsolidated is not None:
    unconsolidated = np.nan_to_num(unconsolidated, nan=0.0, posinf=0.0, neginf=0.0)

# HK deliberately retains NaN.
hk_soil[~np.isfinite(hk_soil)] = np.nan

# ============================================================
# REMOVE NEGATIVE THICKNESS VALUES
# ============================================================
soil[soil < 0] = 0
gst[gst < 0] = 0
if unconsolidated is not None:
    unconsolidated[unconsolidated < 0] = 0

# ============================================================
# APPLY INLAND AND DOMAIN MASKS
# ============================================================
final_arrays = {}   # file name -> array, everything that can be created with the available inputs

# GEBCO remains unmasked
final_arrays["gebco_10km.tif"] = gebco.copy()

# Inland-based processing (needs the inland polygon, NOT in the STAC)
if require("soil_thickness_10km.tif, recharge_10km.tif, hk_soil_10km.tif", "inland"):
    inland_mask_bool = inland_mask_array == 1

    recharge_filled = np.where(inland_mask_bool, recharge, 0.0)
    soil_filled = np.where(inland_mask_bool, soil, 0.0)
    hk_soil_filled = np.where(inland_mask_bool, hk_soil, np.nan)

    # APPLY SOIL/HK DEPENDENCY: HK only where soil thickness > 0 m
    soil_filled = np.maximum(soil_filled, 0.0)
    hk_soil_filled = np.where(soil_filled > 0, hk_soil_filled, np.nan)

    # RECHARGE UNIT CONVERSION: mm/year -> m/day
    recharge_filled = recharge_filled / 1000.0 / 365.0

    final_arrays["soil_thickness_10km.tif"] = soil_filled
    final_arrays["recharge_10km.tif"] = recharge_filled
    final_arrays["hk_soil_10km.tif"] = hk_soil_filled

# Domain-based processing (needs the domain polygon, NOT in the STAC)
if require("gst_10km.tif", "domain"):
    domain_mask_bool = domain_mask_array == 1

    final_arrays["gst_10km.tif"] = np.where(domain_mask_bool, gst, 0.0)

    if require("unconsolidated_10km.tif", "domain", "unconsolidated"):
        final_arrays["unconsolidated_10km.tif"] = np.where(domain_mask_bool, unconsolidated, 0.0)

    # PROCESS ARIDITY INDEX
    # Apply domain mask, fill missing values with zero, convert units (divided by 10,000).
    if require("aridity_index_10km.tif", "domain", "aridity"):
        aridity_filled = np.where(domain_mask_bool, aridity, 0.0)
        aridity_filled = np.nan_to_num(aridity_filled, nan=0.0, posinf=0.0, neginf=0.0)
        final_arrays["aridity_index_10km.tif"] = aridity_filled / 10000.0

# ============================================================
# BASIC ARRAY CHECKS
# ============================================================
for name, arr in final_arrays.items():
    assert arr.shape == (HEIGHT, WIDTH), (
        f"{name} does not match the target grid: {arr.shape}"
    )

print("\nAll processed rasters match the target grid:", list(final_arrays))


# ============================================================
# SAVE FINAL CONTINUOUS RASTERS
# ============================================================

def save_clean_raster(data, filename, nodata=FINAL_NODATA):
    """
    Save a single-band float32 raster on the fixed model grid.

    NaNs and infinite values are replaced with the declared
    nodata value before writing.
    """

    output_path = os.path.join(OUT_DIR, filename)

    arr = np.asarray(data, dtype=np.float32).copy()

    arr[~np.isfinite(arr)] = nodata

    with rasterio.open(
        output_path,
        "w",
        driver="GTiff",
        height=HEIGHT,
        width=WIDTH,
        count=1,
        dtype="float32",
        crs=TARGET_CRS,
        transform=TRANSFORM,
        nodata=nodata,
        compress="lzw"
    ) as dst:
        dst.write(arr, 1)

    print(f"Saved: {output_path}")

    return output_path


for filename, arr in final_arrays.items():
    save_clean_raster(arr, filename)

In [ ]:
# ============================================================
# VALIDATION STATISTICS
# ============================================================

def print_stats(name, arr, units=""):
    """Print statistics for valid, finite cells."""

    valid = np.asarray(arr)
    valid = valid[np.isfinite(valid)]

    print(f"\n{name} {f'({units})' if units else ''}")
    print("-" * 40)

    if valid.size == 0:
        print("No valid cells.")
        return

    print(f"Valid cells: {valid.size:,}")
    print(f"Min:         {valid.min():.6g}")
    print(f"Max:         {valid.max():.6g}")
    print(f"Mean:        {valid.mean():.6g}")


for filename, name, units in [
    ("gebco_10km.tif", "GEBCO", "m"),
    ("recharge_10km.tif", "Recharge", "m/day"),
    ("soil_thickness_10km.tif", "Soil thickness", "m"),
    ("unconsolidated_10km.tif", "Unconsolidated sediment thickness", "m"),
    ("gst_10km.tif", "GST", "m"),
    ("aridity_index_10km.tif", "Aridity index", ""),
]:
    if filename in final_arrays:
        print_stats(name, final_arrays[filename], units)

if "hk_soil_10km.tif" in final_arrays:
    print(
        "\nCells with zero soil but positive HK:",
        int(
            np.sum(
                (soil_filled == 0)
                & np.isfinite(hk_soil_filled)
                & (hk_soil_filled > 0)
            )
        )
    )
    print("HK cells with missing values:", int(np.isnan(hk_soil_filled).sum()))

# ============================================================
# CHECK ALL SAVED RASTERS
# ============================================================
raster_outputs = [
    "gebco_10km.tif",
    "soil_thickness_10km.tif",
    "recharge_10km.tif",
    "gst_10km.tif",
    "unconsolidated_10km.tif",
    "hk_soil_10km.tif",
    "aridity_index_10km.tif",
    "domain_10km.tif",
    "inland_10km.tif",
    "lakes_10km.tif",
]

print("\nFINAL RASTER VALIDATION")
print("=" * 60)

for filename in raster_outputs:

    path = os.path.join(OUT_DIR, filename)

    if not os.path.exists(path):
        print(f"\n{filename}: not created (see the NOT CREATED messages above)")
        continue

    with rasterio.open(path) as src:

        print(f"\n{filename}")
        print(f"  CRS:       {src.crs}")
        print(f"  Size:      {src.width} x {src.height}")
        print(f"  Transform: {src.transform}")
        print(f"  Nodata:    {src.nodata}")

        assert src.crs.to_string() == TARGET_CRS
        assert src.width == WIDTH
        assert src.height == HEIGHT
        assert src.transform == TRANSFORM

print("\nHarmonization finished.")
print("Output directory:", OUT_DIR)

### 4) Assign HK to GEO UNIT "Consolidated Sediments", based on GLiM

**Source:** STAC collection `glim` (column `Litho`). The polygons of the Africa area are read, reprojected to EPSG:3857 and saved once as `GLIM\glim_epsg3857_clipped.gpkg` (replaces `glim_epsg3857_clipped.shp`); the rest of the script is unchanged. No missing input.

In [ ]:
# This script is built to assign HK values to different geology based on GLIM classification and convert the shapefile into a raster that is harmonized
from shapely.geometry import box

# INPUT: GLiM from the STAC, saved locally (reused in sections 5 and 6)
GLIM_PATH = os.path.join(PROCESSED_DIR, "GLIM", "glim_epsg3857_clipped.gpkg")
if REFRESH_LOCAL_COPIES or not os.path.exists(GLIM_PATH):
    read_stac_vector("glim").to_crs("EPSG:3857").to_file(GLIM_PATH, driver="GPKG")
    print("Saved:", GLIM_PATH)

input_shp = GLIM_PATH
output_raster = os.path.join(INPUTS_5KM, "hk_consolidated_5km_xxyy.tif")

attribute = "Litho"  # <-- use full code column

# extent
xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 5000    # original 5 km


# LOOKUP TABLE (HK in m/day)
hk_lookup = {
    "ev__": 0.01,      # evaporites (Domenico, 1990)
    "mt__": 0.01,      # metamorphites (Freeze)
    "mtam": 0.01,
    "mtgr": 0.01,
    "mtpu": 0.01,
    "mtpy": 0.01,
    "pa__": 0.01,      # acidic plutonic (Heath)
    "pb__": 0.01,      # basic plutonic (Heath)
    "pbam": 0.01,
    "pi__": 0.01,      # intermediate plutonic (Heath)
    "pipy": 0.01,
    "scam": 0.1,       # carbonate sedimentary (Heath), mafic-metamorphic mentioned
    "scmx": 0.5,              # mixed grained
    "scpu": 0.2,              # pure carbonate
    "scsh": 0.1,              # fine grained
    "scss": 1,                # coarse grained
    "smmx": 0.5,       # mixed sedimentary (Heath)
    "smpy": 0.1,              # pyroclastics mentioned
    "smsh": 0.1,              # fine grained
    "smss": 1,                # coarse
    "ssmx": 0.5,       # siliciclastic sedimentary (Heath) mixed
    "sspy": 0.1,             # pyroclastics mentioned
    "sssh": 0.1,             # fine
    "ssss": 1,               # coarse
    "va__": 0.01,   # acidic volcanic
    "vapy": 0.01,
    "vb__": 0.01,   # basic volcanic
    "vbpu": 0.01,
    "vbpy": 0.01,
    "vi__": 0.01,   # intermediate volcanic
    "viam": 0.01,
    "vipy": 0.01,
}
bedrock_k = 0.01
default_k = 0.01


# LOAD + REPROJECT
gdf = gpd.read_file(input_shp).to_crs("EPSG:3857")

# CLIP
bbox = gpd.GeoDataFrame(
    geometry=[box(xmin, ymin, xmax, ymax)],
    crs="EPSG:3857"
)

gdf = gpd.clip(gdf, bbox)

# EXTRACT xxyy FROM xxyyzz
gdf["Litho_xxyy"] = (
    gdf[attribute]
    .astype(str)
    .str.lower()
    .str[:4]   # <-- to only get xxyy from the classification
)

# ASSIGN HK
# water bodies, no-data and unconsolidated sediments are treated like hard rock because below them is bedrock
def map_k(val):
    # unconsolidated sediments -> assign default bedrock HK
    if val.startswith("su") or val.startswith("wb") or val.startswith("nd"):
        return bedrock_k
    return hk_lookup.get(val, default_k)

gdf["HK"] = gdf["Litho_xxyy"].apply(map_k)

# GRID
width = int((xmax - xmin) / res)
height = int((ymax - ymin) / res)
transform = from_origin(xmin, ymax, res, res)

# RASTERIZE
shapes = [(geom, val) for geom, val in zip(gdf.geometry, gdf["HK"])]

hk_raster = rasterize(
    shapes,
    out_shape=(height, width),
    transform=transform,
    fill=-9999,
    dtype="float32"
)

# SAVE
with rasterio.open(
    output_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=-9999,
    compress="lzw"
) as dst:
    dst.write(hk_raster, 1)

# STATS
hk_valid = hk_raster[hk_raster > 0]

print("HK min:", float(hk_valid.min()))
print("HK max:", float(hk_valid.max()))
print("Remaining gaps:", np.sum(hk_raster == -9999))
print("Saved:", output_raster)

### 5) Assign HK to GEO UNIT "Unconsolidated Sediments", based on GUM

**Source:** STAC collection `gum` (class code in column `XX`, as in the original) and `glim` (backup). GUM is saved once as `GUM\GUM_epsg3857_clipped.gpkg`.

The final merge uses the land mask `inland_5km.tif`. The original notebook does not create it (it was prepared outside the notebook), so it is rasterized here from the inland polygon, which is **not in the STAC**. Without the inland polygon the GUM and GLiM HK rasters are created, but not the merged `hk_unconsolidated_5km.tif`.

Note: the original lookup has no value for the GUM class `Gt`, which exists in the data. Those polygons get the fallback `-999` (printed as `MISSING classes`), as in the original.

In [ ]:
# This script is built to assign HK values to unconsolidated sediments by GUM and GLIM classification and convert the shapefile into a raster that is harmonized

# INPUT: GUM from the STAC, saved locally
GUM_PATH = os.path.join(PROCESSED_DIR, "GUM", "GUM_epsg3857_clipped.gpkg")
if REFRESH_LOCAL_COPIES or not os.path.exists(GUM_PATH):
    read_stac_vector("gum").to_crs("EPSG:3857").to_file(GUM_PATH, driver="GPKG")
    print("Saved:", GUM_PATH)

input_shp = GUM_PATH
out_raster = os.path.join(PROCESSED_DIR, "GUM", "hk_unconsolidated_5km_GUM.tif")

# attribute field with codes like "Au", "Cu", ...
attribute = "XX"                                     # <-- column name

# extent (EPSG:3857)
xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 5000  # original 5 km

# LOOKUP TABLE
hk_lookup = {
    "Af": 1,      # alluvial fan                depends on position, from gravel to clay
    "Al": 0.5,    # alluvial-lacustrine depos
    "At": 0.5,    # terrace deposits            < 30m/d
    "Au": 1,      # undifferentiated alluvial   0.8-800 m/d
    "Cu": 1,      # colluvial
    "Ed": 10,      # dunes                       Coastal dunes 25 m/d, desert dunes 9.3 m/d,
    "El": 1,       # loess                       0.0001-1.7 m/d
    "Er": 1,       # loess derivates
    "Eu": 0.5,    # aeolian
    "Iy": 0.5,    # pyroclasts                  0.0008-8 m/d
    "Lu": 0.5,    # lacustrine                  0.0008-8 m/d
    "Mu": 0.5,    # marine                      0.01-8 m/d
    "Op": 0.5,    # peat                        0.001-13 m/d
    "Ou": 0.5,    # organic
    "Pg": 0.1,     # gypsum
    "Ps": 0.1,     # salt
    "Pu": 0.1,     # evaporites
    "Us": 10,      # sediments undiff
    "Wl": 0,      # water
    "Wu": 0,      # water
    "Yu": 10,      # coastal dunes
    "Zu": 1       # anthropogenic
}

default_k = -999  # fallback if class missing

# LOAD + REPROJECT
gdf = gpd.read_file(input_shp).to_crs("EPSG:3857")

# CLIP TO EXTENT
bbox = gpd.GeoDataFrame(
    geometry=[box(xmin, ymin, xmax, ymax)],
    crs="EPSG:3857"
)

gdf = gpd.clip(gdf, bbox)

# remove water
gdf = gdf[~gdf[attribute].isin(["Wl", "Wu"])]

# CHECK MISSING CLASSES
unique_classes = set(gdf[attribute].unique())
defined_classes = set(hk_lookup.keys())

missing_classes = unique_classes - defined_classes

print("MISSING classes:", missing_classes)

# ASSIGN HK TO EACH POLYGON
def map_k(val):
    return hk_lookup.get(val, default_k)

gdf["HK"] = gdf[attribute].apply(map_k)

# CREATE GRID
width = int((xmax - xmin) / res)
height = int((ymax - ymin) / res)

transform = from_origin(xmin, ymax, res, res)

# RASTERIZE (DIRECTLY HK)
shapes = [(geom, value) for geom, value in zip(gdf.geometry, gdf["HK"])]

hk_raster = rasterize(
    shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0.0,
    dtype="float32"
)

# PRINT STATS
hk_valid = hk_raster[hk_raster > 0]  # ignore background (0)

if hk_valid.size > 0:
    print("HK min (m/day):", float(hk_valid.min()))
    print("HK max (m/day):", float(hk_valid.max()))
else:
    print("No valid HK values found.")
print("HK min (including zeros):", float(np.min(hk_raster)))

# SAVE RASTER
with rasterio.open(
    out_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=-9999,
    compress="lzw"
) as dst:
    dst.write(hk_raster, 1)

print("HK raster saved:", out_raster)

In [ ]:
# =================================================================================================================
#                    Rasterize Unconsolidated Sedis from GliM as backup for missing GUM areas
# =================================================================================================================

# INPUT (GLiM from the STAC, saved in section 4)
input_shp = GLIM_PATH
output_raster = os.path.join(PROCESSED_DIR, "GLIM", "hk_uncon_5km_GLIM.tif")

attribute = "Litho"

# extent (same as your model grid)
xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 10000   # (the original comment says 5 km, the value is 10 km; it is aligned to the GUM grid in the merge)

# HK LOOKUP (UNCONSOLIDATED ONLY)
hk_lookup = {
    "suad": 0.5, #alluvial dep
    "suds": 10,  #dunes
    "sumx": 1,   #mixed
    "suor": 0.5, #organic
    "supu": 0.1, #pure carbonate
    "sush": 0.5,
    "suss": 1,
}

default_k = 0.001  # important -> keeps gaps clean

# LOAD
gdf = gpd.read_file(input_shp).to_crs("EPSG:3857")

# CLIP TO MODEL DOMAIN
bbox = gpd.GeoDataFrame(
    geometry=[box(xmin, ymin, xmax, ymax)],
    crs="EPSG:3857"
)

gdf = gpd.clip(gdf, bbox)

# CLEAN + EXTRACT UNCONSOLIDATED
gdf = gdf[gdf[attribute].notna()]

# normalize text
gdf[attribute] = gdf[attribute].astype(str).str.lower()

# extract xxyy (first 4 letters)
gdf["Litho_xxyy"] = gdf[attribute].str[:4]

# keep ONLY unconsolidated (su**)
gdf = gdf[gdf["Litho_xxyy"].str.startswith("su")]

# ASSIGN HK
def map_k(val):
    return hk_lookup.get(val, default_k)

gdf["HK"] = gdf["Litho_xxyy"].apply(map_k)

# remove anything without HK
gdf = gdf[~gdf["HK"].isna()]

# GRID
width = int((xmax - xmin) / res)
height = int((ymax - ymin) / res)
transform = from_origin(xmin, ymax, res, res)

# RASTERIZE
shapes = [(geom, val) for geom, val in zip(gdf.geometry, gdf["HK"])]

hk_raster = rasterize(
    shapes,
    out_shape=(height, width),
    transform=transform,
    fill=np.nan,
    dtype="float32"
)

# SAVE
with rasterio.open(
    output_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=-9999,
    compress="lzw"
) as dst:
    dst.write(hk_raster, 1)

# STATS
hk_valid = hk_raster[~np.isnan(hk_raster)]

print("HK min:", float(hk_valid.min()))
print("HK max:", float(hk_valid.max()))
print("Cells filled:", hk_valid.size)
print("Saved:", output_raster)

In [ ]:
# ===============================================================================================================
#                                 Merge Glim and GUM results for Unconsolidated Sediments
# ===============================================================================================================
gum_path = os.path.join(PROCESSED_DIR, "GUM", "hk_unconsolidated_5km_GUM.tif")
glim_path = os.path.join(PROCESSED_DIR, "GLIM", "hk_uncon_5km_GLIM.tif")
consolidated_path = os.path.join(INPUTS_5KM, "hk_consolidated_5km_xxyy.tif")
landmask_path = os.path.join(INPUTS_5KM, "inland_5km.tif")
out_path = os.path.join(INPUTS_5KM, "hk_unconsolidated_5km.tif")

# land mask on the 5 km grid: rasterized from the inland polygon (NOT in the STAC)
if require("inland_5km.tif, hk_unconsolidated_5km.tif", "inland"):
    res = 5000
    width = int((xmax - xmin) / res)
    height = int((ymax - ymin) / res)
    transform_5km = from_origin(xmin, ymax, res, res)
    inland_gdf = load_vector(INLAND_SHP, "Inland polygons")
    inland_5km = rasterize(
        [(geom, 1) for geom in inland_gdf.geometry if geom is not None and not geom.is_empty],
        out_shape=(height, width), transform=transform_5km, fill=0, dtype="uint8"
    )
    write_tif(landmask_path, inland_5km, transform_5km, "EPSG:3857", 255)

    # READ GUM (reference grid)
    with rasterio.open(gum_path) as gum_src:
        gum = gum_src.read(1).astype("float32")
        profile = gum_src.profile
        gum_nodata = gum_src.nodata

    # FUNCTION: normalize to NaN
    def to_nan(arr, nodata):
        out = arr.copy()
        if nodata is not None:
            out[out == nodata] = np.nan
        out[out == 0] = np.nan                   # only if 0 = no data
        return out

    gum_nan = to_nan(gum, gum_nodata)

    # READ + ALIGN GLiM
    with rasterio.open(glim_path) as glim_src:
        glim_raw = glim_src.read(1).astype("float32")
        glim_nodata = glim_src.nodata

        glim = np.full_like(gum, np.nan, dtype="float32")

        reproject(
            source=glim_raw,
            destination=glim,
            src_transform=glim_src.transform,
            src_crs=glim_src.crs,
            dst_transform=profile["transform"],
            dst_crs=profile["crs"],
            resampling=Resampling.nearest
        )

    glim_nan = to_nan(glim, glim_nodata)

    # MERGE (GUM priority)
    merged = np.where(~np.isnan(gum_nan), gum_nan, glim_nan)

    # READ + ALIGN CONSOLIDATED
    with rasterio.open(consolidated_path) as cons_src:
        cons_raw = cons_src.read(1).astype("float32")
        cons_nodata = cons_src.nodata

        cons = np.full_like(gum, np.nan, dtype="float32")

        reproject(
            source=cons_raw,
            destination=cons,
            src_transform=cons_src.transform,
            src_crs=cons_src.crs,
            dst_transform=profile["transform"],
            dst_crs=profile["crs"],
            resampling=Resampling.nearest
        )

    cons_nan = to_nan(cons, cons_nodata)

    # READ + ALIGN LAND MASK
    with rasterio.open(landmask_path) as land_src:
        land_raw = land_src.read(1)

        land = np.full_like(gum, np.nan, dtype="float32")

        reproject(
            source=land_raw,
            destination=land,
            src_transform=land_src.transform,
            src_crs=land_src.crs,
            dst_transform=profile["transform"],
            dst_crs=profile["crs"],
            resampling=Resampling.nearest
        )

    # land = 1, ocean = 0
    land_mask = land == 1

    # DEFAULT FILL (STRICT)
    default_k = 0.1

    no_uncon = np.isnan(merged)
    no_cons = np.isnan(cons_nan)

    # FINAL CONDITION
    fill_mask = no_uncon & no_cons & land_mask

    merged[fill_mask] = default_k

    # CONVERT NaN -> -9999
    nodata_val = -9999
    merged_out = merged.copy()
    merged_out[np.isnan(merged_out)] = nodata_val

    # SAVE
    profile.update(dtype="float32", nodata=nodata_val, compress="lzw")

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(merged_out, 1)

    # STATS
    valid = merged_out != nodata_val

    print("Cells (total):", merged_out.size)
    print("Cells (valid):", int(valid.sum()))
    print("Cells filled with default:", int(np.sum(fill_mask)))
    print("HK min:", float(merged_out[valid].min()))
    print("HK max:", float(merged_out[valid].max()))
    print("Saved:", out_path)

### 6) Assign POROSITY to all GEOLOGICAL UNITS (Soil, UnconS, ConsS)

**Sources:** soil from the sand/silt/clay rasters of section 1; unconsolidated sediments from the GLHYMPS GeoPackage created in section 3c (STAC `glhymps`, column `Porosity_x`; the original read a `GLHYMPS_Africa_3857.shp`); consolidated sediments from GLiM (section 4). No missing input.

In [ ]:
# ==========================================================================================
#                             PREPROCESS POROSITY - GEO LAYER 1 (SOIL)
# ==========================================================================================

# PATHS
input_dir = os.path.join(PROCESSED_DIR, "SOILGRIDS")
out_dir = os.path.join(PROCESSED_DIR, "GLHYMPS")

sand_file = os.path.join(input_dir, "sand_weighted_5km_3857.tif")
silt_file = os.path.join(input_dir, "silt_weighted_5km_3857.tif")
clay_file = os.path.join(input_dir, "clay_weighted_5km_3857.tif")

output_file = os.path.join(out_dir, "porosity_layer1_5km.tif")

# LOAD RASTERS
print("Loading SoilGrids fractions...")

with rasterio.open(sand_file) as src:
    sand = src.read(1)
    profile = src.profile.copy()

with rasterio.open(silt_file) as src:
    silt = src.read(1)

with rasterio.open(clay_file) as src:
    clay = src.read(1)

# CLEAN DATA
sand = np.nan_to_num(sand)
silt = np.nan_to_num(silt)
clay = np.nan_to_num(clay)

# convert percentages if necessary
if sand.max() > 1:

    sand /= 100
    silt /= 100
    clay /= 100

# EFFECTIVE POROSITY
print("Assigning effective porosity...")

porosity = np.zeros(sand.shape, dtype="float32")

# Sand-dominated
mask = sand >= 0.70
porosity[mask] = 0.4                  # medium sand 0.16 - 0.46

# Silt-dominated
mask = silt >= 0.50
porosity[mask] = 0.3                  # silt 0.01 - 0.39

# Clay-dominated
mask = clay >= 0.40
porosity[mask] = 0.1                  # clay 0.01 - 0.18

# Mixed soils
mask = porosity == 0
porosity[mask] = 0.25

# SAVE
profile.update(
    dtype="float32",
    compress="lzw",
    nodata=-9999
)

with rasterio.open(output_file, "w", **profile) as dst:
    dst.write(porosity, 1)

# STATISTICS
valid = porosity[porosity >= 0]
print("\n=============================")
print("Layer 1 Porosity Statistics")
print("=============================")
print("Min :", float(valid.min()))
print("Max :", float(valid.max()))
print("Mean:", float(valid.mean()))
print("Std :", float(valid.std()))
print()
print("Sand cells :", np.sum(sand >= 0.70))
print("Clay cells :", np.sum(clay >= 0.40))
print("Silt cells :", np.sum(silt >= 0.50))
print("Mixed cells:", np.sum(porosity == 0.25))
print()
print("Saved:")
print(output_file)

In [ ]:
#==========================================================================================
#                       PREPROCESS POROSITY - GEO LAYER 2 Uncons Sedis (GLHYMPS)
#==========================================================================================

# INPUT: GLHYMPS (STAC) cropped and saved in section 3c
input_shp = os.path.join(INPUTS_10KM, "GLHYMPS_Africa_3857.gpkg")
output_raster = os.path.join(PROCESSED_DIR, "GLHYMPS", "porosity_layer2_5km.tif")

# MODEL GRID
xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 5000

width = int((xmax-xmin)/res)
height = int((ymax-ymin)/res)

transform = from_origin(xmin, ymax, res, res)

# LOAD
print("Loading GLHYMPS...")

gdf = gpd.read_file(input_shp)

gdf = gdf.to_crs("EPSG:3857")

# FILTER TO MODEL EXTENT (FAST AND ROBUST)
print("Filtering to model extent...")

gdf = gdf.cx[xmin:xmax, ymin:ymax]

print(f"Remaining polygons: {len(gdf)}")

# CONVERT POROSITY
print("Converting porosity...")
# GLHYMPS stores porosity multiplied by 100
gdf["POR"] = gdf["Porosity_x"] / 100.0

# remove impossible values
gdf.loc[gdf["POR"] < 0,"POR"] = np.nan
gdf.loc[gdf["POR"] > 1,"POR"] = np.nan

# Enforce minimum effective porosity
gdf.loc[gdf["POR"] < 0.01, "POR"] = 0.01

# RASTERIZE
print("Rasterizing...")
shapes = [
    (geom,val)
    for geom,val in zip(gdf.geometry,gdf["POR"])
    if np.isfinite(val)
]

porosity = rasterize(
    shapes,
    out_shape=(height,width),
    transform=transform,
    fill=-9999,
    dtype="float32"
)

# SAVE
with rasterio.open(
    output_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=-9999,
    compress="lzw"
) as dst:

    dst.write(porosity,1)

# STATISTICS
valid = porosity[porosity!=-9999]
print()
print("========== Layer 2 ==========")
print(f"Min  : {valid.min():.3f}")
print(f"Max  : {valid.max():.3f}")
print(f"Mean : {valid.mean():.3f}")
print(f"Std  : {valid.std():.3f}")
print(f"Cells: {valid.size}")
print()
print("Saved:")
print(output_raster)

In [ ]:
# ==========================================================================================
#                       PREPROCESS POROSITY - GEO LAYER 3 (Consolid Sedis via GLiM)
# ==========================================================================================

# INPUT (GLiM from the STAC, saved in section 4)
input_shp = GLIM_PATH
output_raster = os.path.join(PROCESSED_DIR, "GLHYMPS", "porosity_layer3_5km.tif")

attribute = "Litho"

# MODEL GRID
xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 5000

width = int((xmax-xmin)/res)
height = int((ymax-ymin)/res)

transform = from_origin(xmin, ymax, res, res)

# EFFECTIVE POROSITY LOOKUP
por_lookup = {

    # evaporites
    "ev__": 0.01,

    # metamorphic
    "mt__": 0.01,
    "mtam": 0.01,
    "mtgr": 0.01,
    "mtpu": 0.01,
    "mtpy": 0.01,

    # plutonic
    "pa__": 0.01,
    "pb__": 0.01,
    "pbam": 0.01,
    "pi__": 0.01,
    "pipy": 0.01,

    # carbonate sedimentary
    "scam": 0.15,
    "scmx": 0.15,
    "scpu": 0.1,
    "scsh": 0.1,
    "scss": 0.2,

    # mixed sedimentary
    "smmx": 0.15,
    "smpy": 0.1,
    "smsh": 0.1,
    "smss": 0.25,

    # siliciclastic sedimentary
    "ssmx": 0.15,
    "sspy": 0.1,
    "sssh": 0.1,
    "ssss": 0.25,

    # volcanic
    "va__": 0.05,
    "vapy": 0.05,
    "vb__": 0.05,
    "vbpu": 0.05,
    "vbpy": 0.05,
    "vi__": 0.05,
    "viam": 0.05,
    "vipy": 0.05,
}

default_porosity = 0.05

# LOAD
print("Loading GLIM...")

gdf = gpd.read_file(input_shp).to_crs("EPSG:3857")

# FILTER TO MODEL EXTENT
print("Filtering...")

gdf = gdf.cx[xmin:xmax, ymin:ymax]

print("Remaining polygons:", len(gdf))

# EXTRACT XXYY
gdf["Litho_xxyy"] = (
    gdf[attribute]
    .astype(str)
    .str.lower()
    .str[:4]
)

# ASSIGN POROSITY
def map_porosity(val):

    if val.startswith("su"):
        return 0.25       # unconsolidated sediments

    if val.startswith("wb"):
        return 0.1        # below water

    if val.startswith("nd"):
        return default_porosity

    return por_lookup.get(val, default_porosity)

gdf["POR"] = gdf["Litho_xxyy"].apply(map_porosity)


# RASTERIZE
print("Rasterizing...")
shapes = [
    (geom,val)
    for geom,val in zip(gdf.geometry,gdf["POR"])
]

porosity = rasterize(
    shapes,
    out_shape=(height,width),
    transform=transform,
    fill=-9999,
    dtype="float32"
)

# SAVE
with rasterio.open(
    output_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=-9999,
    compress="lzw"
) as dst:

    dst.write(porosity,1)

# STATISTICS
valid = porosity[porosity != -9999]

print()
print("====================================")
print("Layer 3 Effective Porosity")
print("====================================")
print(f"Minimum : {valid.min():.3f}")
print(f"Maximum : {valid.max():.3f}")
print(f"Mean    : {valid.mean():.3f}")
print(f"Std     : {valid.std():.3f}")
print(f"Cells   : {valid.size}")
print()
print("Saved:")
print(output_raster)

In [ ]:
# =====================================================================================
#                     RESAMPLE POROSITY RASTERS from 5 km  -> 10 km
#
# Uses bilinear interpolation.
# =====================================================================================

# INPUT FOLDER
input_dir = os.path.join(PROCESSED_DIR, "GLHYMPS")

# INPUT FILES
input_files = [
    "porosity_layer1_5km.tif",
    "porosity_layer2_5km.tif",
    "porosity_layer3_5km.tif"
]

# OUTPUT GRID
xmin = -2145000
ymin = -4440000
xmax = 5965000
ymax = 4645000

res = 10000

width = int((xmax - xmin) / res)
height = int((ymax - ymin) / res)

dst_transform = from_origin(
    xmin,
    ymax,
    res,
    res
)

# LOOP
for file in input_files:

    print(f"\nProcessing {file}")

    input_path = os.path.join(input_dir, file)
    output_path = os.path.join(
        input_dir,
        file.replace("_5km", "_10km")
    )

    with rasterio.open(input_path) as src:
        profile = src.profile.copy()
        profile.update(
            width=width,
            height=height,
            transform=dst_transform,
            compress="lzw"
        )

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=dst_transform,
                dst_crs=src.crs,
                resampling=Resampling.bilinear
            )

    print("Saved:", output_path)
print("\nFinished.")

### 7) ASSIGNING GEO/HYDRAULIC PROPERTIES -> NUMERICAL LAYERS (for > 3 numerical layers)

The script reads its inputs from the `10_x_10_km\6_ml` folder. The original notebook does not create some of those files (they were prepared outside the notebook), so the cell 7a below **creates them from the results of the previous sections**:

| File in `6_ml` | Made from |
|---|---|
| `inland_10km`, `soil_thickness_10km`, `unconsolidated_10km`, `gst_10km`, `hk_soil_10km` | copied from `10_x_10_km` (section 3) |
| `hk_consolidated_10km_xxyy` | `hk_consolidated_5km_xxyy` (section 4), resampled to 10 km (average, as the soil HK) |
| `hk_unconsolidated_10km` | `hk_unconsolidated_5km` (section 5), resampled to 10 km (average) |
| `porosity_soil_10km`, `porosity_uncon_10km`, `porosity_cons_10km` | `porosity_layer1/2/3_10km` (section 6), copied |
| `seabed_litho_epsg3857` | read from the STAC (`seabed_lithology`, codes 1-13) and reprojected to the model grid inside 7b |

The resampling of the two HK rasters is an assumption (the original notebook does not show how they were made).

**Missing inputs:** the numerical layers (7b) need `inland_10km`, `gst_10km` and `unconsolidated_10km`, so they need the inland polygon, the domain polygon and the unconsolidated thickness (none of them in the STAC). The seabed properties (7a) are created in any case.

In [ ]:
# ------------------------------------------------------------------
# 7a) Prepare the 6_ml folder (not in the original notebook) and the seabed lithology
# ------------------------------------------------------------------
input_dir = ML6_DIR
output_dir = ML6_DIR

# copy the section 3 results that exist
for filename in ["inland_10km.tif", "soil_thickness_10km.tif", "unconsolidated_10km.tif", "gst_10km.tif", "hk_soil_10km.tif"]:
    src_file = os.path.join(INPUTS_10KM, filename)
    if os.path.exists(src_file):
        shutil.copy(src_file, os.path.join(input_dir, filename))
        print("Copied:", filename)
    else:
        print(f"NOT CREATED: {filename} in 6_ml  <- not created in section 3 (see the messages there)")

# porosity: section 6 names -> names used by this section
for layer_file, new_name in [
    ("porosity_layer1_10km.tif", "porosity_soil_10km.tif"),
    ("porosity_layer2_10km.tif", "porosity_uncon_10km.tif"),
    ("porosity_layer3_10km.tif", "porosity_cons_10km.tif"),
]:
    shutil.copy(os.path.join(PROCESSED_DIR, "GLHYMPS", layer_file), os.path.join(input_dir, new_name))
    print("Copied:", new_name)

# HK of consolidated / unconsolidated sediments: 5 km -> 10 km grid
HK_RESAMPLING = Resampling.average


def resample_to_10km(src_path, dst_path, resampling):
    with rasterio.open(src_path) as src:
        out = np.full((HEIGHT, WIDTH), -9999, dtype="float32")
        reproject(
            source=rasterio.band(src, 1),
            destination=out,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=TRANSFORM,
            dst_crs=TARGET_CRS,
            dst_nodata=-9999,
            resampling=resampling,
        )
    write_tif(dst_path, out, TRANSFORM, TARGET_CRS, -9999)


resample_to_10km(
    os.path.join(INPUTS_5KM, "hk_consolidated_5km_xxyy.tif"),
    os.path.join(input_dir, "hk_consolidated_10km_xxyy.tif"),
    HK_RESAMPLING,
)
if os.path.exists(os.path.join(INPUTS_5KM, "hk_unconsolidated_5km.tif")):
    resample_to_10km(
        os.path.join(INPUTS_5KM, "hk_unconsolidated_5km.tif"),
        os.path.join(input_dir, "hk_unconsolidated_10km.tif"),
        HK_RESAMPLING,
    )
else:
    print("NOT CREATED: hk_unconsolidated_10km.tif  <- hk_unconsolidated_5km.tif was not created in section 5 (missing input: Africa_inland.shp)")

# ------------------------------------------------------------------
# Model grid (the original read it from soil_thickness_10km.tif)
# ------------------------------------------------------------------
profile = dict(driver="GTiff", height=HEIGHT, width=WIDTH, count=1, crs=TARGET_CRS, transform=TRANSFORM)
model_transform = TRANSFORM
model_crs = TARGET_CRS
model_width = WIDTH
model_height = HEIGHT

# Set number of layers here:
nlay = 6

# ------------------------------------------------------------------
# Seabed lithology: STAC raster (EPSG:4326, codes 1-13) -> model grid
# ------------------------------------------------------------------
seabed_href = stac_href("seabed_lithology")
print("Seabed lithology COG:", seabed_href)

with rasterio.open(seabed_href) as src:
    seabed_src = src.read(1).astype(np.float32)
    seabed_lithology = np.full(
        (model_height, model_width),
        np.nan,
        dtype=np.float32)
    reproject(
        source=seabed_src,
        destination=seabed_lithology,
        src_transform=src.transform,
        src_crs=src.crs,
        src_nodata=src.nodata,
        dst_transform=model_transform,
        dst_crs=model_crs,
        dst_nodata=np.nan,
        resampling=Resampling.nearest
    )

valid_codes = seabed_lithology[
    np.isfinite(seabed_lithology)
]
print("Seabed grid shape:", seabed_lithology.shape)
print("Seabed lithology codes:", np.unique(valid_codes))

# -----------------------------------------------------------------
# Seabed lithology properties
# -----------------------------------------------------------------
# code : [HK (m/day), porosity (-), VK/HK ratio]
#
# 1  Gravel and coarser
# 2  Sand
# 3  Silt
# 4  Clay
# 5  Calcareous ooze
# 6  Radiolarian ooze
# 7  Diatom ooze
# 8  Sponge spicules
# 9  Mixed calcareous/siliceous ooze
# 10 Shells and coral fragments
# 11 Ash and volcanic sand/gravel
# 12 Siliceous mud
# 13 Fine-grained calcareous sediment

seabed_properties = {
    1:  {"hk": 10.0,  "por": 0.35, "vk_ratio": 0.1},
    2:  {"hk": 5.0,   "por": 0.40, "vk_ratio": 0.1},
    3:  {"hk": 0.50,  "por": 0.50, "vk_ratio": 0.1},
    4:  {"hk": 0.05,  "por": 0.60, "vk_ratio": 0.1},
    5:  {"hk": 0.1,  "por": 0.60, "vk_ratio": 0.1},
    6:  {"hk": 0.1,  "por": 0.70, "vk_ratio": 0.1},
    7:  {"hk": 0.1,  "por": 0.70, "vk_ratio": 0.1},
    8:  {"hk": 0.1,  "por": 0.70, "vk_ratio": 0.1},
    9:  {"hk": 0.1,  "por": 0.60, "vk_ratio": 0.1},
    10: {"hk": 5.0,  "por": 0.40, "vk_ratio": 0.1},
    11: {"hk": 1.0,   "por": 0.40, "vk_ratio": 0.1},
    12: {"hk": 0.05,  "por": 0.60, "vk_ratio": 0.1},
    13: {"hk": 0.05,  "por": 0.60, "vk_ratio": 0.1},
}

hk_seabed = np.full(seabed_lithology.shape, np.nan, dtype=float)
vk_seabed = np.full(seabed_lithology.shape, np.nan, dtype=float)
por_seabed = np.full(seabed_lithology.shape, np.nan, dtype=float)

for code, props in seabed_properties.items():

    mask = seabed_lithology == code
    hk_seabed[mask] = props["hk"]
    vk_seabed[mask] = (props["hk"] * props["vk_ratio"])
    por_seabed[mask] = props["por"]


# ------------------------------------------------------------------
# Save raster function
# ------------------------------------------------------------------

def save_raster(array, filename):

    out_profile = profile.copy()

    out_profile.update(
        dtype=rasterio.float32,
        count=1,
        nodata=np.nan,
        compress='lzw'
    )

    output_fp = os.path.join(
        output_dir,
        filename
    )

    with rasterio.open(
        output_fp,
        'w',
        **out_profile
    ) as dst:

        dst.write(
            array.astype(np.float32),
            1
        )


save_raster(hk_seabed, "hk_seabed.tif")
save_raster(vk_seabed, "vk_seabed.tif")
save_raster(por_seabed, "porosity_seabed.tif")
print("Saved seabed properties in", output_dir)

In [ ]:
# ------------------------------------------------------------------
# 7b) Numerical layers: assigns the hk, vk and porosity for all numerical model layers
# (dominant geology per cell), for 6 numerical layers in this case
# ------------------------------------------------------------------
# =================================================================================
#                        CASE 3: 6 numerical layers
# =================================================================================

if require("hk/vk/porosity/dominant_geology layers 1-6", "domain", "inland", "unconsolidated"):
    # ------------------------------------------------------------------
    # Load mask
    # ------------------------------------------------------------------

    with rasterio.open(os.path.join(input_dir, 'inland_10km.tif')) as src:
        inland = src.read(1)


    # ------------------------------------------------------------------
    # Load ORIGINAL geological thicknesses
    # These must be the original thicknesses WITHOUT numerical minimums
    # ------------------------------------------------------------------

    with rasterio.open(os.path.join(input_dir, 'soil_thickness_10km.tif')) as src:
        soil_thk_orig = src.read(1).astype(float)

    with rasterio.open(os.path.join(input_dir, 'unconsolidated_10km.tif')) as src:
        uncon_thk_orig = src.read(1).astype(float)

    with rasterio.open(os.path.join(input_dir, 'gst_10km.tif')) as src:
        gst_thk_orig = src.read(1).astype(float)

    # safety check
    print("Model grid shape:", soil_thk_orig.shape)

    # ------------------------------------------------------------------
    # Load ORIGINAL geological HK
    # ------------------------------------------------------------------

    with rasterio.open(os.path.join(input_dir, 'hk_soil_10km.tif')) as src:
        hk_soil = src.read(1).astype(float)

    with rasterio.open(os.path.join(input_dir, 'hk_unconsolidated_10km.tif')) as src:
        hk_uncon = src.read(1).astype(float)

    with rasterio.open(os.path.join(input_dir, 'hk_consolidated_10km_xxyy.tif')) as src:
        hk_cons = src.read(1).astype(float)


    # ------------------------------------------------------------------
    # Load ORIGINAL geological porosity
    # ------------------------------------------------------------------

    with rasterio.open(os.path.join(input_dir, 'porosity_soil_10km.tif')) as src:
        por_soil = src.read(1).astype(float)

    with rasterio.open(os.path.join(input_dir, 'porosity_uncon_10km.tif')) as src:
        por_uncon = src.read(1).astype(float)

    with rasterio.open(os.path.join(input_dir, 'porosity_cons_10km.tif')) as src:
        por_cons = src.read(1).astype(float)


    # ------------------------------------------------------------------
    # Clean original thicknesses
    # ------------------------------------------------------------------

    soil_thk_orig = np.nan_to_num(soil_thk_orig, nan=0.0)
    uncon_thk_orig = np.nan_to_num(uncon_thk_orig, nan=0.0)
    gst_thk_orig = np.nan_to_num(gst_thk_orig, nan=0.0)

    soil_thk_orig = np.maximum(soil_thk_orig, 0.0)
    uncon_thk_orig = np.maximum(uncon_thk_orig, 0.0)

    # Consolidated thickness = total sediment thickness
    # minus soil and unconsolidated thickness
    cons_thk_orig = gst_thk_orig - (
        soil_thk_orig + uncon_thk_orig
    )

    cons_thk_orig = np.maximum(cons_thk_orig, 0.0)


    # ------------------------------------------------------------------
    # Calculate geological VK from geological HK
    # ------------------------------------------------------------------

    vk_soil = hk_soil * 0.1
    vk_uncon = hk_uncon * 0.1
    vk_cons = hk_cons * 0.1

    # -----------------------------------------------------------------
    # Numerical 6-layer geometry
    #
    # L1: soil interval, minimum 10 m
    #
    # L2 + L3: original unconsolidated interval,
    #          minimum total numerical thickness = 20 m
    #
    # L4 + L5: original consolidated interval,
    #          minimum total numerical thickness = 20 m
    #
    # L6: basal numerical layer below mapped geology.
    #     This layer is intended to act as the basal constant-salinity
    #     reservoir during transport simulations.

    num_thk1 = np.maximum(soil_thk_orig, 10.0)

    num_uncon_total = np.maximum(uncon_thk_orig, 20.0)
    num_thk2 = num_uncon_total / 2.0
    num_thk3 = num_uncon_total / 2.0

    num_cons_total = np.maximum(cons_thk_orig, 20.0)
    num_thk4 = num_cons_total / 2.0
    num_thk5 = num_cons_total / 2.0

    # Dedicated basal layer
    basal_thickness = 100.0

    num_thk6 = np.full_like(
        soil_thk_orig,
        basal_thickness,
        dtype=float
    )

    num_thickness = np.stack([
        num_thk1,
        num_thk2,
        num_thk3,
        num_thk4,
        num_thk5,
        num_thk6
    ])


    # ------------------------------------------------------------------
    # Geological interfaces measured as depth below surface
    # ------------------------------------------------------------------

    geo_soil_top = np.zeros_like(soil_thk_orig)
    geo_soil_bottom = soil_thk_orig

    geo_uncon_top = geo_soil_bottom
    geo_uncon_bottom = geo_uncon_top + uncon_thk_orig

    geo_cons_top = geo_uncon_bottom
    geo_cons_bottom = geo_cons_top + cons_thk_orig


    # ------------------------------------------------------------------
    # Numerical layer interfaces measured as depth below surface
    # ------------------------------------------------------------------

    num_top = np.zeros_like(num_thickness)
    num_bottom = np.zeros_like(num_thickness)

    running_depth = np.zeros_like(soil_thk_orig)

    for k in range(nlay):

        num_top[k] = running_depth

        num_bottom[k] = (
            running_depth + num_thickness[k]
        )

        running_depth = num_bottom[k]


    # ------------------------------------------------------------------
    # Function to calculate vertical overlap between:
    #
    # numerical layer [num_top, num_bottom]
    # geological unit [geo_top, geo_bottom]
    #
    # Returns overlap thickness in metres.
    # ------------------------------------------------------------------

    def overlap_thickness(num_top, num_bottom, geo_top, geo_bottom):

        overlap = (
            np.minimum(num_bottom, geo_bottom)
            - np.maximum(num_top, geo_top)
        )

        return np.maximum(overlap, 0.0)


    # ------------------------------------------------------------------
    # Output arrays
    # ------------------------------------------------------------------

    nrow, ncol = soil_thk_orig.shape

    hk_layers = np.zeros((nlay, nrow, ncol), dtype=float)
    vk_layers = np.zeros((nlay, nrow, ncol), dtype=float)
    por_layers = np.zeros((nlay, nrow, ncol), dtype=float)

    # dominant geology:
    #
    # 0 = undefined / outside model
    # 1 = soil
    # 2 = unconsolidated
    # 3 = consolidated
    # 4 = bedrock
    # 5 = seabed sediment

    # This is useful to save so you can inspect exactly which geological
    # material was assigned to every numerical cell.
    dominant_geology = np.zeros(
        (nlay, nrow, ncol),
        dtype=np.int8
    )

    # ------------------------------------------------------------------
    # Bedrock default properties
    # ------------------------------------------------------------------

    hk_bedrock = 0.05
    vk_bedrock = hk_bedrock * 0.1
    por_bedrock = 0.01


    # ------------------------------------------------------------------
    # Assign properties according to DOMINANT GEOLOGICAL MATERIAL
    # ------------------------------------------------------------------

    for k in range(nlay):

        # Calculate how many metres of each geological material
        # occur inside this numerical layer
        soil_overlap = overlap_thickness(
            num_top[k],
            num_bottom[k],
            geo_soil_top,
            geo_soil_bottom
        )

        uncon_overlap = overlap_thickness(
            num_top[k],
            num_bottom[k],
            geo_uncon_top,
            geo_uncon_bottom
        )

        cons_overlap = overlap_thickness(
            num_top[k],
            num_bottom[k],
            geo_cons_top,
            geo_cons_bottom
        )

        # Stack geological overlaps
        overlaps = np.stack([
            soil_overlap,
            uncon_overlap,
            cons_overlap
        ])

        # Find geology with largest overlap
        dominant = np.argmax(overlaps, axis=0)
        max_overlap = np.max(overlaps, axis=0)

        # Save as 1/2/3 rather than 0/1/2
        dominant_geology[k] = dominant + 1

        # Assign complete material properties according to dominant unit
        mask_soil = dominant == 0
        mask_uncon = dominant == 1
        mask_cons = dominant == 2

        hk_layers[k][mask_soil] = hk_soil[mask_soil]
        vk_layers[k][mask_soil] = vk_soil[mask_soil]
        por_layers[k][mask_soil] = por_soil[mask_soil]

        hk_layers[k][mask_uncon] = hk_uncon[mask_uncon]
        vk_layers[k][mask_uncon] = vk_uncon[mask_uncon]
        por_layers[k][mask_uncon] = por_uncon[mask_uncon]

        hk_layers[k][mask_cons] = hk_cons[mask_cons]
        vk_layers[k][mask_cons] = vk_cons[mask_cons]
        por_layers[k][mask_cons] = por_cons[mask_cons]

        # If this numerical layer lies below all mapped geology,
        # assign bedrock rather than accepting np.argmax -> soil.
        no_overlap = max_overlap <= 0.0

        hk_layers[k][no_overlap] = hk_bedrock
        vk_layers[k][no_overlap] = vk_bedrock
        por_layers[k][no_overlap] = por_bedrock
        dominant_geology[k][no_overlap] = 4


    # ------------------------------------------------------------------
    # Offshore section
    # ------------------------------------------------------------------

    inland_mask = inland == 1
    ocean_mask = ~inland_mask

    valid_seabed = (
        ocean_mask
        & np.isfinite(seabed_lithology)
        & (seabed_lithology >= 1)
        & (seabed_lithology <= 13)
        & np.isfinite(hk_seabed)
        & np.isfinite(vk_seabed)
        & np.isfinite(por_seabed)
    )

    # Assign artificial/default properties to the complete offshore column first.
    # Valid seabed properties will subsequently overwrite Layers 1 and 2.
    for k in range(5):

        hk_layers[k][ocean_mask] = 0.1
        vk_layers[k][ocean_mask] = 0.05
        por_layers[k][ocean_mask] = 0.1

        dominant_geology[k][ocean_mask] = 4

    # Layer 6 = basal bedrock everywhere
    hk_layers[5][ocean_mask] = hk_bedrock
    vk_layers[5][ocean_mask] = vk_bedrock
    por_layers[5][ocean_mask] = por_bedrock
    dominant_geology[5][ocean_mask] = 4

    # Apply seabed lithology properties to offshore Layers 1 and 2.
    for k in [0, 1]:
        hk_layers[k][valid_seabed] = hk_seabed[valid_seabed]
        vk_layers[k][valid_seabed] = vk_seabed[valid_seabed]
        por_layers[k][valid_seabed] = por_seabed[valid_seabed]
        dominant_geology[k][valid_seabed] = 5


    # New code 5 = seabed sediment
    dominant_geology[0][valid_seabed] = 5
    dominant_geology[1][valid_seabed] = 5

    # ------------------------------------------------------------------
    # Save HK, VK and porosity
    # ------------------------------------------------------------------

    for k in range(nlay):

        layer = k + 1

        save_raster(
            hk_layers[k],
            f"hk_layer{layer}.tif"
        )

        save_raster(
            vk_layers[k],
            f"vk_layer{layer}.tif"
        )

        save_raster(
            por_layers[k],
            f"porosity_layer{layer}.tif"
        )

    # ------------------------------------------------------------------
    # Save dominant-geology rasters
    #
    # 0 = outside model
    # 1 = soil
    # 2 = unconsolidated
    # 3 = consolidated
    # 4 = bedrock
    # 5 = seabed sediment
    # ------------------------------------------------------------------

    for k in range(nlay):

        layer = k + 1

        save_raster(
            dominant_geology[k],
            f"dominant_geology_layer{layer}.tif"
        )


    print("Finished dominant-geology preprocessing.")
    print()
    print("Dominant geology codes:")
    print("0 = undefined / outside model")
    print("1 = soil")
    print("2 = unconsolidated")
    print("3 = consolidated")
    print("4 = bedrock")
    print("5 = seabed sediment")

### 8) HYDRO RIVERS -> Creating the RIV PACKAGE with CRIV, Stage, Botm elevation

This script preprocesses HydroRIVERS and creates the MODFLOW RIV input using the approach of calculating geometric river properties per reach and then summing or averaging them per cell (c_geom). A climate dependent CRIV is also calculated (c_climate) and the higher CRIV is chosen.

**Source:** STAC collection `hydrorivers` (item `hydrorivers_af`, column `DIS_AV_CMS`). The GEBCO 1 km raster (EPSG:3857) comes from section 3a.

**Missing inputs:** this section needs the domain polygon, the inland polygon, the aridity index and `vk_layer1` from section 7b (so also the unconsolidated thickness). None of them is in the STAC, so nothing is created until they are in `Data\missing_inputs`.

Changes with respect to the original:
- the original reads `aridity_index_clipped_3857.tif`; here the 10 km raster of section 3 (`aridity_index_10km.tif`) is used and multiplied by 10000 again, because the thresholds below (6500, 5000, 2000, 300) are expressed in the original index units (x 10000);
- all paths point to the folders of this notebook (the original mixes `...\IMOD\Inputs` and `...\Inputs`).

The calculation can take a long time (overlay of all rivers with the 10 km grid).

In [ ]:
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

# In
dem_fp = os.path.join(INPUTS_10KM, "gebco_10km.tif")
dem_1km_fp = GEBCO_3857_PATH
soil_fp = os.path.join(INPUTS_10KM, "soil_thickness_10km.tif")
domain_fp = MISSING_PATHS["domain"]
vk_fp = os.path.join(ML6_DIR, "vk_layer1.tif")           # vk_layer1 from section 7
inland_fp = os.path.join(INPUTS_10KM, "inland_10km.tif")
aridity_fp = os.path.join(INPUTS_10KM, "aridity_index_10km.tif")

# Out
riv_package_fp = os.path.join(INPUTS_10KM, "riv_package_10x10_c5.0.txt")
plot_dir = RIVER_PLOT_DIR
os.makedirs(plot_dir, exist_ok=True)

# domain
xmin, ymin, xmax, ymax = -2145000, -4440000, 5965000, 4645000
cell_size = 10000

if require("riv_package_10x10_c5.0.txt", "domain", "inland", "unconsolidated", "aridity"):
    from rasterstats import zonal_stats

    print("Loading paths done")

    # LOAD + REPROJECT + CLIP TO DOMAIN
    gdf = read_stac_vector("hydrorivers").to_crs(epsg=3857)

    domain = gpd.read_file(domain_fp).to_crs(epsg=3857)
    gdf = gpd.clip(gdf, domain)                          # clip rivers to domain

    # Use all rivers
    gdf = gdf[gdf["DIS_AV_CMS"] > 0]

    print("Load, reproject, clip done")

    # ----------------------------- CREATE GRID -------------------------------------
    cols = np.arange(xmin, xmax, cell_size)
    rows = np.arange(ymin, ymax, cell_size)

    grid_cells = [box(x, y, x+cell_size, y+cell_size) for x in cols for y in rows]
    grid = gpd.GeoDataFrame(geometry=grid_cells, crs="EPSG:3857")
    grid["cell_id"] = np.arange(len(grid))

    grid = gpd.overlay(grid, domain, how="intersection")      # mask the grid with domain

    # measure length of river per cell
    intersections = gpd.overlay(gdf, grid, how="intersection")
    intersections["length_m"] = intersections.geometry.length

    # -------------------------------- LOAD rasters ---------------------------------
    with rasterio.open(dem_fp) as src:
        top_arr = src.read(1)
        transform = src.transform

    with rasterio.open(soil_fp) as src:
        soil_arr = src.read(1)

    with rasterio.open(vk_fp) as src:
        vk_arr = src.read(1)

    with rasterio.open(inland_fp) as src:
        inland_arr = src.read(1)

    with rasterio.open(aridity_fp) as src:
        # section 3 divides the index by 10000; the thresholds below use the original units
        aridity_arr = src.read(1) * 10000.0

    # apply minimum thickness = 10 m (same as MODFLOW model)
    soil_arr = np.nan_to_num(soil_arr, nan=0.0)
    soil_arr[soil_arr < 10] = 10.0

    nrow, ncol = top_arr.shape

    print("load rasters done")

    # ----------------  SAMPLE TOP, THICKNESS, VK PER INTERSECTIONS ------------------------

    def sample_raster(x, y, arr, transform):
        col, row = ~transform * (x, y)
        col, row = int(col), int(row)
        if 0 <= row < arr.shape[0] and 0 <= col < arr.shape[1]:
            return arr[row, col]
        else:
            return np.nan

    intersections["top"] = [
        sample_raster(geom.centroid.x,
                      geom.centroid.y,
                      top_arr,
                      transform)
        for geom in intersections.geometry
    ]

    intersections["vk_layer1"] = [
        sample_raster(geom.centroid.x,
                      geom.centroid.y,
                      vk_arr,
                      transform)
        for geom in intersections.geometry
    ]

    # sample inland mask
    intersections["inland"] = [

        sample_raster(
            geom.centroid.x,
            geom.centroid.y,
            inland_arr,
            transform
        )
        for geom in intersections.geometry
    ]

    grid["soil_thickness"] = [
        sample_raster(geom.centroid.x, geom.centroid.y, soil_arr, transform)
        for geom in grid.geometry
    ]

    grid["top"] = [
        sample_raster(
            geom.centroid.x,
            geom.centroid.y,
            top_arr,
            transform
        )
        for geom in grid.geometry
    ]

    grid["aridity"] = [
        sample_raster(
            geom.centroid.x,
            geom.centroid.y,
            aridity_arr,
            transform
        )
        for geom in grid.geometry
    ]

    print("Sampling done")


    # REMOVE RIVERS OUTSIDE INLAND
    before_count = len(intersections)
    # keep only inland river segments
    intersections = intersections[
        intersections["inland"] == 1
    ].copy()
    after_count = len(intersections)
    removed = before_count - after_count
    print("\nOFFSHORE RIVER FILTER")
    print(f"River segments before filter: {before_count}")
    print(f"River segments after filter:  {after_count}")
    print(f"Removed offshore segments:    {removed}")
    print("removing rivers outside inland done")

    # Use aridity index for fractions for COND
    def river_fraction_from_aridity(ai):
        if np.isnan(ai):    # NaN
            return 0.01

        if ai > 6500:       # humid
            return 0.15
        elif ai > 5000:     # subhumid
            return 0.10
        elif ai > 2000:     # semi-arid
            return 0.05
        elif ai > 300:      # arid
            return 0.02
        else:               # hyperarid
            return 0.005

    # Define C values for CRIV ~ Area_surface_water / c
    def resistance_from_aridity(ai):
        if np.isnan(ai):     # NaN
            return 20
        if ai > 6500:        # humid
            return 10
        elif ai > 5000:      # subhumid
            return 15
        elif ai > 2000:      # semi-arid
            return 20
        elif ai > 300:       # arid
            return 40
        else:                # hyperarid
            return 100

    # ----------------------- COMPUTATIONS PER RIVER SEGMENT-----------------------------------------

    #  bottom, thickness [m]
    grid["bottom"] = grid["top"] - grid["soil_thickness"]
    grid["thickness"] = grid["soil_thickness"]

    # elevation class
    def elev_class(z):
        if z > 1000:
            return "mountain"
        elif z > 300:
            return "transition"
        else:
            return "lowland"

    intersections["elev_class"] = intersections["top"].apply(elev_class)


    # width, depth (from q, terrain classification)
    def width_depth_factors(elev_class):
        if elev_class == "mountain":
            width_factor = 0.4      # narrower
            depth_factor = 2.5      # deeper
        elif elev_class == "transition":
            width_factor = 0.7
            depth_factor = 1.5
        else:  # lowlands
            width_factor = 1.0      # wider
            depth_factor = 1.0      # shallower
        return width_factor, depth_factor
    widths = []
    depths = []

    for _, row in intersections.iterrows():

        Q = row["DIS_AV_CMS"]

        wf, df = width_depth_factors(row["elev_class"])

        W = (7.2 * (Q ** 0.5)) * wf
        D = (0.27 * (Q ** 0.4)) * df

        W = np.clip(W, 1, 700)
        D = np.clip(D, 0.5, 50)

        widths.append(W)
        depths.append(D)

    intersections["width"] = widths
    intersections["depth"] = depths

    print(intersections["width"].describe())
    print(intersections["depth"].describe())

    # riverbed thickness
    def assign_M(elev_class):
        if elev_class == "mountain":
            return 0.1
        elif elev_class == "transition":
            return 10
        else:             # lowlands
            return 50

    intersections["thickness_rb"] = (
        intersections["elev_class"]
        .apply(assign_M)
    )

    # K_rb conceptual (after terrain) without q class
    def assign_K(elev_class): # K_rb value in [m/d]
        if elev_class == "mountain":
            return 0.1
        elif elev_class == "transition":
            return 2
        else:                 # lowlands
            return 5

    intersections["Krb_conceptual"] = intersections.apply(
        lambda r: assign_K(
            r["elev_class"]
        ),
        axis=1
    )

    intersections["Krb"] = np.minimum(
        intersections["Krb_conceptual"],
        intersections["vk_layer1"]
    )
    print("K_rb done")
    # ------------------ AGGREGATE TO CELL GRID ------------------

    agg = intersections.groupby("cell_id").agg({
        "length_m": "max",                           #use max instead of sum!
        "depth": "mean",
        "Krb": "mean",
        "thickness_rb": "mean"
    }).reset_index()

    # calculate effective width using root-sum-square
    width_eff = (
        intersections
        .groupby("cell_id")["width"]
        .apply(lambda x: np.sqrt(np.sum(x**2)))
        .reset_index(name="width")
    )
    agg = agg.merge(width_eff, on="cell_id")

    grid = grid.merge(
        agg,
        on="cell_id",
        how="left"
    )

    # For computation of C_climate
    cell_area = 10000 * 10000

    grid["river_fraction"] = (
        grid["aridity"]
        .apply(river_fraction_from_aridity)
    )
    grid["c_climate"] = (
        grid["aridity"]
        .apply(resistance_from_aridity)
    )
    # Compute C_climate
    grid["C_climate"] = (
        cell_area
        * grid["river_fraction"]
        / grid["c_climate"]
    )

    # constrain length at 10km for dense river networks for aggregation per cell
    grid["length_m"] = np.minimum(
        grid["length_m"],
        10000
    )

    print("Aggregation done")


    # ----------------------- COMPUTE STAGE AND BED ELEVATION -----------------------
    # stage [m]
    # -> 5th percentile of lowest elevation in cell or top-thk/2
    stats = zonal_stats(
        grid,
        dem_1km_fp,
        stats=None,
        add_stats={"p5": lambda x: np.nanpercentile(np.asarray(x).astype(float).copy(), 5 )},
        nodata=-9999
    )
    grid["stage"] = [s["p5"] for s in stats]
    # constraints in top and bottom:
    grid["stage"] = np.minimum( grid["stage"], grid["top"] - 0.1)   # max DEM - 0.1 m
    grid["stage"] = np.maximum( grid["stage"], grid["top"] - 50)    # min DEM - 50

    # bed elevation
    grid["bed_elev"] = grid["stage"] - grid["depth"]
    grid["bed_elev"] = np.minimum(grid["bed_elev"],  grid["stage"] - 0.1) # ensure bed minimum below -50.1m

    print("bed elevation done")

    # --------------------- Calculate CRIV conductance [m2/day] --------------
    grid["C_geom"] = (
        grid["Krb"]
        * grid["length_m"]
        * grid["width"]
    ) / grid["thickness_rb"]
    print(grid["C_geom"].describe())

    print("computation C_geom done")

    # mask river cells
    grid["has_river"] = ~grid["C_geom"].isna()
    grid["conductance"] = np.nan
    river_cells = grid["has_river"]

    grid.loc[river_cells, "conductance"] = np.maximum(
        grid.loc[river_cells, "C_geom"],
        grid.loc[river_cells, "C_climate"]
    )

    print("Cells with top < -50 and river:")
    print(
        np.sum(
            (grid["conductance"].notna()) &
            (grid["top"] < -50)
        )
    )

    # diagnostics
    diag_fp = os.path.join(
        plot_dir,
        "river_conductance_diagnostics.csv"
    )
    grid[
    [
        "cell_id",
        "aridity",
        "river_fraction",
        "c_climate",
        "C_climate",
        "C_geom",
        "conductance"
    ]
    ].to_csv(
        diag_fp,
        index=False
    )
    print("Saved:", diag_fp)

    # summary stats
    summary_fp = os.path.join(
        plot_dir,
        "river_conductance_summary.csv"
    )
    grid[
    [
        "aridity",
        "river_fraction",
        "c_climate",
        "C_climate",
        "C_geom",
        "conductance"
    ]
    ].describe().to_csv(
        summary_fp
    )
    print("Saved:", summary_fp)

    # investigate high conductance areas
    highC = grid.nlargest(20, "conductance")
    print(
        highC[
            [
                "conductance",
                "length_m",
                "width",
                "Krb",
                "thickness_rb",
                "top",
            ]
        ]
    )
    print("Finished computation. RIV package will be written next.")

    # Remove RIV from Ocean
    grid["inland"] = [
        sample_raster(
            geom.centroid.x,
            geom.centroid.y,
            inland_arr,
            transform
        )
        for geom in grid.geometry
    ]
    river_before = grid["conductance"].notna().sum()

    grid.loc[
        grid["inland"] != 1,
        "conductance"
    ] = np.nan

    river_after = grid["conductance"].notna().sum()

    print("Removed river cells:", river_before-river_after)

    # ----------------------------- WRITE RIV PACKAGE --------------------------
    with open(riv_package_fp, "w") as f:
        for _, row in grid.dropna(subset=["conductance"]).iterrows():

            layer = 1
            r = int((ymax - row.geometry.centroid.y) / cell_size)
            c = int((row.geometry.centroid.x - xmin) / cell_size)

            f.write(f"{layer} {r} {c} "
                    f"{row['stage']} {row['conductance']} {row['bed_elev']}\n")

    print("write package done:", riv_package_fp)

### Summary: what was created and what is still missing

In [ ]:
# Every expected output with the missing inputs it depends on
EXPECTED = []


def expect(path, *needs):
    EXPECTED.append((path, needs))


for p in [os.path.join(SOILGRIDS_DIR, f) for f in ["clay_weighted_5km_3857.tif", "sand_weighted_5km_3857.tif", "silt_weighted_5km_3857.tif", "hk_soil_cosby_5km.tif"]]:
    expect(p)
expect(os.path.join(PROCESSED_DIR, "GST", "GST_python_epsg4326.tif"))
for f in ["Africa_GEBCO.tif", "Africa_GEBCO_3857.tif", "Africa_SOIL.tif"]:
    expect(os.path.join(PROCESSED_DIR, "unharmonized", f))

expect(os.path.join(INPUTS_10KM, "domain_10km.tif"), "domain")
expect(os.path.join(INPUTS_10KM, "inland_10km.tif"), "inland")
expect(os.path.join(INPUTS_10KM, "lakes_10km.tif"), "domain")
expect(os.path.join(INPUTS_10KM, "GLHYMPS_Africa_3857.gpkg"))
expect(os.path.join(INPUTS_10KM, "gebco_10km.tif"))
expect(os.path.join(INPUTS_10KM, "soil_thickness_10km.tif"), "inland")
expect(os.path.join(INPUTS_10KM, "recharge_10km.tif"), "inland")
expect(os.path.join(INPUTS_10KM, "hk_soil_10km.tif"), "inland")
expect(os.path.join(INPUTS_10KM, "gst_10km.tif"), "domain")
expect(os.path.join(INPUTS_10KM, "unconsolidated_10km.tif"), "domain", "unconsolidated")
expect(os.path.join(INPUTS_10KM, "aridity_index_10km.tif"), "domain", "aridity")

expect(os.path.join(INPUTS_5KM, "hk_consolidated_5km_xxyy.tif"))
expect(os.path.join(PROCESSED_DIR, "GUM", "hk_unconsolidated_5km_GUM.tif"))
expect(os.path.join(PROCESSED_DIR, "GLIM", "hk_uncon_5km_GLIM.tif"))
expect(os.path.join(INPUTS_5KM, "inland_5km.tif"), "inland")
expect(os.path.join(INPUTS_5KM, "hk_unconsolidated_5km.tif"), "inland")

for layer in (1, 2, 3):
    expect(os.path.join(PROCESSED_DIR, "GLHYMPS", f"porosity_layer{layer}_5km.tif"))
    expect(os.path.join(PROCESSED_DIR, "GLHYMPS", f"porosity_layer{layer}_10km.tif"))

for f in ["hk_seabed.tif", "vk_seabed.tif", "porosity_seabed.tif", "hk_consolidated_10km_xxyy.tif"]:
    expect(os.path.join(ML6_DIR, f))
expect(os.path.join(ML6_DIR, "hk_unconsolidated_10km.tif"), "inland")
for k in range(1, 7):
    for prefix in ["hk_layer", "vk_layer", "porosity_layer", "dominant_geology_layer"]:
        expect(os.path.join(ML6_DIR, f"{prefix}{k}.tif"), "domain", "inland", "unconsolidated")

expect(os.path.join(INPUTS_10KM, "riv_package_10x10_c5.0.txt"), "domain", "inland", "unconsolidated", "aridity")

created, not_created = [], []
for path, needs in EXPECTED:
    if os.path.exists(path):
        created.append(path)
    else:
        missing = [MISSING_FILES[n][0] for n in needs if not have(n)]
        not_created.append((path, missing))

print(f"Created: {len(created)} of {len(EXPECTED)} expected files\n")
print("NOT created:")
for path, missing in not_created:
    reason = "missing input: " + ", ".join(missing) if missing else "run the corresponding section"
    print(f"  {os.path.relpath(path, MODEL_DIR)}  <- {reason}")

still_missing = sorted({m for _, miss in not_created for m in miss})
if still_missing:
    print("\nFiles to add to", MISSING_DIR)
    for key, (name, description) in MISSING_FILES.items():
        if name in still_missing:
            print(f"  {name}  ({description})")